# PropRAG vs GraphRAG vs BaseRAG — 2WikiMultiHopQA Benchmark

Self-contained benchmark notebook for **Google Colab** (free T4 GPU).

| Component | Details |
|---|---|
| **LLM** | `openai/gpt-oss-20b` loaded in **4-bit NF4** (bitsandbytes) — no GGUF/Koboldcpp needed |
| **Embedder** | `BAAI/bge-large-en-v1.5` via sentence-transformers |
| **Dataset** | 2WikiMultiHopQA (PropRAG-main repository clone or HuggingFace Hub) |

### Quick-start
1. Set `HF_TOKEN` in the **Configuration** cell (Cell 5).
2. Go to `Runtime → Run all`.
3. Results appear inline at the bottom and are saved to `/content/data/benchmark/`.

> **Tip:** For a fast timing pilot, set `PILOT = 5`.

## Section 0 — GPU Check

In [ ]:
import subprocess, sys

r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
if r.returncode == 0:
    info = r.stdout.strip()
    print(f"GPU detected: {info}")
    vram = int(info.split(",")[1].strip().split()[0])
    if vram < 14000:
        print(f"WARNING: only {vram} MB VRAM — gpt-oss-20b 4-bit needs ~14 GB. Consider Colab Pro (A100).")
    else:
        print(f"OK: {vram} MB VRAM — sufficient for 4-bit gpt-oss-20b.")
else:
    print("No GPU found. Go to Runtime -> Change runtime type -> GPU (T4).")

## Section 1 — Install Dependencies
> First run takes ~5 minutes. Subsequent runs skip already-installed packages.

In [ ]:
%%capture install_output
# Core ML stack
!pip install -q "transformers>=4.48.0" accelerate "bitsandbytes>=0.43.0" sentencepiece protobuf

# Benchmark dependencies
!pip install -q "openai>=1.30" "sentence-transformers>=2.7" "python-igraph>=0.11" \
               numpy pandas pyarrow "tqdm>=4.66" matplotlib huggingface_hub datasets

# Lightweight OpenAI-compatible API server
!pip install -q fastapi "uvicorn[standard]" pydantic

print("All packages installed successfully.")

In [ ]:
import os

PROPRAG_MAIN = "/content/PropRAG_main"

if not os.path.isdir(PROPRAG_MAIN):
    print("Cloning PropRAG-main repository from GitHub...")
    ret = os.system(f"git clone --quiet https://github.com/ReLink-Inc/PropRAG.git {PROPRAG_MAIN}")
    if ret != 0:
        raise RuntimeError("git clone failed — check network connectivity.")
    print(f"Cloned to {PROPRAG_MAIN}")
else:
    print(f"Already present: {PROPRAG_MAIN}")

# Confirm proprag_poc module is present
poc_init = os.path.join(PROPRAG_MAIN, "proprag_poc", "config.py")
if not os.path.isfile(poc_init):
    raise ImportError(
        f"proprag_poc/config.py not found inside {PROPRAG_MAIN}.\n"
        "The GitHub repository layout may have changed — inspect the clone and adjust PROPRAG_MAIN."
    )
print(f"proprag_poc found: {PROPRAG_MAIN}/proprag_poc/")

## Section 2 — Configuration
> **Edit this cell to configure the evaluation run.**

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────

# HuggingFace access token.
# Required for openai/gpt-oss-20b (you must accept the model license first:
# https://huggingface.co/openai/gpt-oss-20b)
HF_TOKEN = ""   # e.g. "hf_xxxxxxxxxxxxxxxxxxxx"

# Model to load
MODEL_ID = "openai/gpt-oss-20b"

# Embedding model (drop to BAAI/bge-base-en-v1.5 to save ~900 MB RAM)
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5"

# ── Benchmark knobs ───────────────────────────────────────────────────────────
N_QUESTIONS   = 50        # Total questions to evaluate
SEED          = 42
PILOT         = None      # Set to e.g. 5 for a quick pilot; None for the full run
SYSTEMS       = ["BaseRAG", "GraphRAG", "PropRAG"]
FORCE_REINDEX = False     # True = rebuild all indexes even if cached
MAKE_CHARTS   = True

# ── Model loading ─────────────────────────────────────────────────────────────
BNB_QUANT_TYPE         = "nf4"       # "nf4" or "fp4"
BNB_COMPUTE_DTYPE      = "float16"   # "float16" or "bfloat16"
BNB_DOUBLE_QUANT       = True
MAX_NEW_TOKENS_DEFAULT = 1500        # Hard cap applied server-side

# ── API server (no need to change) ───────────────────────────────────────────
LLAMA_HOST = "127.0.0.1"
LLAMA_PORT = 5001

print("Configuration loaded.")

## Section 3 — HuggingFace Authentication & Path Setup

In [ ]:
import os, sys

# HuggingFace token
if HF_TOKEN:
    os.environ["HF_TOKEN"]               = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Logged in to HuggingFace Hub.")
    except Exception as e:
        print(f"HF login warning (non-fatal): {e}")
else:
    print("No HF_TOKEN set. Download may fail if gpt-oss-20b is gated.")

# sys.path: proprag_poc lives inside PROPRAG_MAIN
for p in [PROPRAG_MAIN, "/content"]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Env vars read by proprag_poc's POCConfig
os.environ["PROPRAG_LLM_BACKEND"]     = "koboldcpp"   # koboldcpp -> localhost:PORT/v1
os.environ["PROPRAG_LLM_MODEL"]       = MODEL_ID
os.environ["PROPRAG_EMBEDDING_MODEL"] = EMBEDDING_MODEL

print("sys.path and env vars configured.")

## Section 4 — Write Benchmark Package to Colab Disk
> Writes the benchmark source files to `/content/benchmark/`.

In [ ]:
import json, os

# Create directory structure
os.makedirs("/content/benchmark/graphrag", exist_ok=True)

# Load and write all bundled benchmark files
serialized_files = '{"benchmark/__init__.py": "\\"\\"\\"PropRAG vs GraphRAG vs BaseRAG benchmark harness on 2WikiMultiHopQA.\\"\\"\\"\\n", "benchmark/_bootstrap.py": "\\"\\"\\"Put the ``proprag_poc`` package on ``sys.path``.\\n\\n``proprag_poc`` is not pip-installable and uses package-relative imports, so it\\nimports cleanly only when its PARENT directory is importable. This file lives at\\n``PropRAG/PropRAG Testing/benchmark/_bootstrap.py``; ``parents[2]`` is the\\n``PropRAG`` directory that holds ``proprag_poc``. Import this module first from\\nevery entry point.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport sys\\nfrom pathlib import Path\\n\\n_PROPRAG_PARENT = Path(__file__).resolve().parents[2]\\n\\nif str(_PROPRAG_PARENT) not in sys.path:\\n    sys.path.insert(0, str(_PROPRAG_PARENT))\\n\\n# Fail loudly and early if the layout assumption is wrong.\\nif not (_PROPRAG_PARENT / \\"proprag_poc\\" / \\"config.py\\").is_file():\\n    raise ImportError(\\n        f\\"proprag_poc not found next to the project. Expected it at \\"\\n        f\\"{_PROPRAG_PARENT / \'proprag_poc\'}. Keep \'PropRAG Testing\' as a sibling \\"\\n        f\\"of \'proprag_poc\' under the PropRAG directory.\\"\\n    )\\n", "benchmark/bench_config.py": "\\"\\"\\"Benchmark configuration wrapping the reused ``POCConfig``.\\n\\nOne ``BenchmarkConfig`` object drives the whole run. It carries benchmark-only\\nknobs (subset size, GraphRAG parameters, per-call token caps) and knows how to\\nbuild the ``POCConfig`` the reused ``proprag_poc`` library expects, pinned to the\\nlocal Koboldcpp backend and the shared sentence-transformers embedder.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport os\\nfrom dataclasses import dataclass, field\\nfrom typing import Optional, Tuple\\n\\nfrom . import _bootstrap  # noqa: F401 - side effect: proprag_poc on sys.path\\nfrom proprag_poc.config import POCConfig\\n\\n# Default dataset location (sibling PropRAG-main checkout).\\n_DEFAULT_DATASET = os.path.join(\\n    _bootstrap._PROPRAG_PARENT,\\n    \\"PropRAG-main\\",\\n    \\"reproduce\\",\\n    \\"dataset\\",\\n    \\"2wikimultihopqa.json\\",\\n)\\n_DEFAULT_CORPUS = os.path.join(\\n    _bootstrap._PROPRAG_PARENT,\\n    \\"PropRAG-main\\",\\n    \\"reproduce\\",\\n    \\"dataset\\",\\n    \\"2wikimultihopqa_corpus.json\\",\\n)\\n\\n\\n@dataclass\\nclass BenchmarkConfig:\\n    project_dir: str\\n    dataset_path: str = _DEFAULT_DATASET\\n    corpus_path: str = _DEFAULT_CORPUS\\n\\n    # ----- Subset selection -----\\n    n_questions: int = 50\\n    seed: int = 42\\n    pilot: Optional[int] = None\\n\\n    # ----- Shared retrieval / QA -----\\n    recall_ks: Tuple[int, ...] = (2, 5, 10)\\n    retrieval_top_k: int = 10\\n    qa_top_k: int = 5                 # passages fed to the generator (all systems)\\n    qa_max_answer_tokens: int = 512\\n\\n    # ----- Per-call completion caps (runtime control on a CPU-served 20B) -----\\n    extract_max_tokens: int = 1500\\n    summarize_max_tokens: int = 512\\n    report_max_tokens: int = 1000\\n\\n    # ----- GraphRAG knobs -----\\n    gr_entity_types: Tuple[str, ...] = (\\n        \\"person\\", \\"organization\\", \\"location\\", \\"event\\", \\"work\\", \\"date\\", \\"other\\",\\n    )\\n    gr_max_gleanings: int = 0         # Microsoft default is 1; 0 halves index cost (flagged in report)\\n    gr_summarize_desc_threshold: int = 800   # merged-description chars before LLM summarization\\n    gr_min_community_size: int = 2\\n    gr_report_max_input_chars: int = 12000   # ~3000-token cap on community-report context\\n    gr_local_top_entities: int = 10\\n    gr_entity_sim_floor: float = 0.5\\n    gr_dense_blend: float = 0.5\\n\\n    systems: Tuple[str, ...] = (\\"BaseRAG\\", \\"GraphRAG\\", \\"PropRAG\\")\\n\\n    def __post_init__(self):\\n        self.project_dir = os.path.abspath(self.project_dir)\\n\\n    @property\\n    def data_dir(self) -> str:\\n        return os.path.join(self.project_dir, \\"data\\")\\n\\n    def make_poc_config(self) -> POCConfig:\\n        \\"\\"\\"Build the library config, pinned to the local Koboldcpp + bge stack.\\n\\n        ``PROPRAG_*`` env vars still override backend/model/embedder (e.g. drop to\\n        ``BAAI/bge-base-en-v1.5`` under RAM pressure) via ``POCConfig`` itself.\\n        \\"\\"\\"\\n        return POCConfig(\\n            data_dir=self.data_dir,\\n            llm_backend=\\"koboldcpp\\",   # -> http://localhost:5001/v1, gpt-oss-20b-Q4_K_M.gguf\\n            temperature=0.0,\\n            seed=self.seed,\\n            retrieval_top_k=self.retrieval_top_k,\\n            qa_top_k=self.qa_top_k,\\n            llm_max_workers=1,         # Koboldcpp serves one request at a time\\n            strip_reasoning=True,      # honoured by BenchLLMClient (POC leaves it unimplemented)\\n        )\\n", "benchmark/dataset.py": "\\"\\"\\"2WikiMultiHopQA loading, stratified subsetting, and corpus assembly.\\n\\nPure Python (no LLM, no embeddings) so it can be sanity-checked offline via\\n``python -m benchmark.dataset``.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport os\\nimport random\\nfrom dataclasses import dataclass, field\\nfrom typing import Dict, List, Tuple\\n\\n# Canonical type order: fixes largest-remainder tie-breaks and output sorting.\\nTYPE_ORDER = (\\"compositional\\", \\"comparison\\", \\"bridge_comparison\\", \\"inference\\")\\n\\n\\n@dataclass\\nclass BenchQuestion:\\n    qid: str\\n    qtype: str\\n    question: str\\n    answer: str\\n    gold_titles: List[str]\\n    context_titles: List[str]\\n\\n    @property\\n    def gold_answers(self) -> List[str]:\\n        \\"\\"\\"EM/F1 gold list. 2wiki gives one canonical answer per question.\\"\\"\\"\\n        return [self.answer]\\n\\n\\n# --------------------------------------------------------------------- loading\\ndef load_questions(path: str) -> List[BenchQuestion]:\\n    with open(path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        raw = json.load(f)\\n    out: List[BenchQuestion] = []\\n    for q in raw:\\n        gold = list(dict.fromkeys(sf[0] for sf in q.get(\\"supporting_facts\\", [])))\\n        ctx = list(dict.fromkeys(c[0] for c in q.get(\\"context\\", [])))\\n        out.append(\\n            BenchQuestion(\\n                qid=q[\\"_id\\"],\\n                qtype=q[\\"type\\"],\\n                question=q[\\"question\\"],\\n                answer=q[\\"answer\\"],\\n                gold_titles=gold,\\n                context_titles=ctx,\\n            )\\n        )\\n    return out\\n\\n\\n# ------------------------------------------------------------- allocation math\\ndef _largest_remainder(counts: Dict[str, int], n: int) -> Dict[str, int]:\\n    \\"\\"\\"Proportionally allocate ``n`` slots across types; tie-break by TYPE_ORDER.\\n\\n    Clamps each allocation to what the type actually has and redistributes any\\n    shortfall to types with spare capacity (largest remainder first).\\n    \\"\\"\\"\\n    types = [t for t in TYPE_ORDER if t in counts] + [\\n        t for t in counts if t not in TYPE_ORDER\\n    ]\\n    total = sum(counts.values())\\n    if total == 0 or n <= 0:\\n        return {t: 0 for t in types}\\n\\n    raw = {t: counts[t] * n / total for t in types}\\n    alloc = {t: int(raw[t]) for t in types}\\n    order_index = {t: i for i, t in enumerate(types)}\\n\\n    def frac_key(t: str) -> Tuple[float, int]:\\n        return (-(raw[t] - int(raw[t])), order_index[t])\\n\\n    remainder = n - sum(alloc.values())\\n    for t in sorted(types, key=frac_key)[:remainder]:\\n        alloc[t] += 1\\n\\n    # Clamp to availability, then redistribute any shortfall.\\n    for t in types:\\n        alloc[t] = min(alloc[t], counts[t])\\n    shortfall = n - sum(alloc.values())\\n    while shortfall > 0:\\n        spare = [t for t in types if alloc[t] < counts[t]]\\n        if not spare:\\n            break\\n        for t in sorted(spare, key=frac_key):\\n            if shortfall == 0:\\n                break\\n            alloc[t] += 1\\n            shortfall -= 1\\n    return alloc\\n\\n\\ndef _stratified_pick(\\n    qs: List[BenchQuestion], n: int, seed: int\\n) -> List[BenchQuestion]:\\n    by_type: Dict[str, List[BenchQuestion]] = {}\\n    for q in qs:\\n        by_type.setdefault(q.qtype, []).append(q)\\n    counts = {t: len(v) for t, v in by_type.items()}\\n    alloc = _largest_remainder(counts, n)\\n\\n    picked: List[BenchQuestion] = []\\n    rng = random.Random(seed)\\n    for t in sorted(by_type, key=lambda x: TYPE_ORDER.index(x) if x in TYPE_ORDER else 99):\\n        pool = sorted(by_type[t], key=lambda q: q.qid)  # deterministic pool order\\n        picked.extend(rng.sample(pool, alloc[t]))\\n    picked.sort(key=lambda q: (TYPE_ORDER.index(q.qtype) if q.qtype in TYPE_ORDER else 99, q.qid))\\n    return picked\\n\\n\\ndef stratified_subset(qs: List[BenchQuestion], n: int, seed: int) -> List[BenchQuestion]:\\n    return _stratified_pick(qs, n, seed)\\n\\n\\ndef pilot_subset(qs_subset: List[BenchQuestion], k: int, seed: int) -> List[BenchQuestion]:\\n    \\"\\"\\"Stratified ``k``-of-subset. Its questions are a subset of ``qs_subset`` so\\n    every pilot extraction becomes a cache hit during the full run.\\"\\"\\"\\n    return _stratified_pick(qs_subset, k, seed)\\n\\n\\n# ----------------------------------------------------------------- corpus build\\ndef chunk_text(title: str, text: str) -> str:\\n    \\"\\"\\"One searchable chunk per document: the title on its own line, then body.\\"\\"\\"\\n    return f\\"{title}\\\\n{text}\\"\\n\\n\\ndef build_corpus(qs: List[BenchQuestion], corpus_path: str) -> Dict[str, str]:\\n    \\"\\"\\"title -> text for the union of the subset\'s context titles.\\n\\n    Asserts every gold and context title exists in the corpus so Recall@k and QA\\n    context are never silently missing a gold passage.\\n    \\"\\"\\"\\n    with open(corpus_path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        raw = json.load(f)\\n    full: Dict[str, str] = {}\\n    for doc in raw:\\n        title = doc[\\"title\\"]\\n        if title not in full:  # dedupe by title, first wins\\n            full[title] = doc[\\"text\\"]\\n\\n    needed = set()\\n    for q in qs:\\n        needed.update(q.context_titles)\\n        needed.update(q.gold_titles)\\n\\n    missing = sorted(t for t in needed if t not in full)\\n    if missing:\\n        raise ValueError(\\n            f\\"{len(missing)} needed titles absent from corpus, e.g. {missing[:3]}\\"\\n        )\\n    return {t: full[t] for t in sorted(needed)}\\n\\n\\n# -------------------------------------------------------------------- identity\\ndef subset_hash(qs: List[BenchQuestion]) -> str:\\n    joined = \\"|\\".join(sorted(q.qid for q in qs))\\n    return hashlib.sha1(joined.encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef corpus_id(qs: List[BenchQuestion], tag: str) -> str:\\n    return f\\"2wiki_{tag}_{subset_hash(qs)[:10]}\\"\\n\\n\\ndef type_counts(qs: List[BenchQuestion]) -> Dict[str, int]:\\n    counts: Dict[str, int] = {}\\n    for q in qs:\\n        counts[q.qtype] = counts.get(q.qtype, 0) + 1\\n    return counts\\n\\n\\ndef write_manifest(\\n    run_dir: str, qs: List[BenchQuestion], titles: Dict[str, str], seed: int, cfg\\n) -> str:\\n    os.makedirs(run_dir, exist_ok=True)\\n    path = os.path.join(run_dir, \\"manifest.json\\")\\n    manifest = {\\n        \\"seed\\": seed,\\n        \\"n_questions\\": len(qs),\\n        \\"type_counts\\": type_counts(qs),\\n        \\"subset_hash\\": subset_hash(qs),\\n        \\"corpus_size\\": len(titles),\\n        \\"qids\\": [q.qid for q in qs],\\n        \\"dataset_path\\": cfg.dataset_path,\\n        \\"corpus_path\\": cfg.corpus_path,\\n    }\\n    with open(path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(manifest, f, indent=2)\\n    return path\\n\\n\\n# ---------------------------------------------------------------------- sanity\\ndef _main() -> None:\\n    from .bench_config import BenchmarkConfig\\n\\n    cfg = BenchmarkConfig(project_dir=os.path.dirname(os.path.dirname(__file__)))\\n    qs = load_questions(cfg.dataset_path)\\n    print(f\\"loaded {len(qs)} questions; full type counts: {type_counts(qs)}\\")\\n\\n    subset = stratified_subset(qs, cfg.n_questions, cfg.seed)\\n    print(f\\"subset n={len(subset)} type counts: {type_counts(subset)}\\")\\n\\n    pilot = pilot_subset(subset, 10, cfg.seed)\\n    print(f\\"pilot n={len(pilot)} type counts: {type_counts(pilot)}\\")\\n    assert set(p.qid for p in pilot).issubset(set(s.qid for s in subset)), \\"pilot not a subset\\"\\n\\n    titles = build_corpus(subset, cfg.corpus_path)\\n    print(f\\"corpus subset size: {len(titles)} docs (expect ~380-420)\\")\\n\\n    # Gold-title coverage.\\n    all_gold = set()\\n    for q in subset:\\n        all_gold.update(q.gold_titles)\\n    covered = sum(1 for t in all_gold if t in titles)\\n    print(f\\"gold titles: {len(all_gold)}, covered by corpus: {covered}\\")\\n    assert covered == len(all_gold), \\"some gold titles are missing from the corpus\\"\\n    print(f\\"corpus_id: {corpus_id(subset, \'n50\')}\\")\\n    print(\\"dataset sanity OK\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    _main()\\n", "benchmark/evaluation.py": "\\"\\"\\"Answer and retrieval metrics.\\n\\n``normalize_answer`` / EM / F1 are ported verbatim from PropRAG-main\\n(``utils/eval_utils.normalize_answer`` and ``evaluation/qa_eval``), stripped of\\nthe ``BaseMetric`` scaffolding. Recall@k is the standard supporting-title recall.\\nRun ``python -m benchmark.evaluation`` to exercise the inline hand cases.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\nimport string\\nfrom collections import Counter\\nfrom typing import Dict, List, Sequence\\n\\n\\ndef normalize_answer(answer: str) -> str:\\n    \\"\\"\\"Lowercase, drop punctuation, drop a/an/the, collapse whitespace.\\"\\"\\"\\n    def remove_articles(text: str) -> str:\\n        return re.sub(r\\"\\\\b(a|an|the)\\\\b\\", \\" \\", text)\\n\\n    def white_space_fix(text: str) -> str:\\n        return \\" \\".join(text.split())\\n\\n    def remove_punc(text: str) -> str:\\n        exclude = set(string.punctuation)\\n        return \\"\\".join(ch for ch in text if ch not in exclude)\\n\\n    return white_space_fix(remove_articles(remove_punc(answer.lower())))\\n\\n\\ndef em_score(golds: Sequence[str], pred: str) -> float:\\n    \\"\\"\\"Exact match, max over the gold list (MRQA-style).\\"\\"\\"\\n    npred = normalize_answer(pred)\\n    return max((1.0 if normalize_answer(g) == npred else 0.0) for g in golds) if golds else 0.0\\n\\n\\ndef _f1(gold: str, pred: str) -> float:\\n    gold_tokens = normalize_answer(gold).split()\\n    pred_tokens = normalize_answer(pred).split()\\n    common = Counter(pred_tokens) & Counter(gold_tokens)\\n    num_same = sum(common.values())\\n    if num_same == 0:\\n        return 0.0\\n    precision = num_same / len(pred_tokens)\\n    recall = num_same / len(gold_tokens)\\n    return 2 * precision * recall / (precision + recall)\\n\\n\\ndef f1_score(golds: Sequence[str], pred: str) -> float:\\n    \\"\\"\\"Token-overlap F1, max over the gold list.\\"\\"\\"\\n    return max((_f1(g, pred) for g in golds), default=0.0)\\n\\n\\ndef recall_at_k(\\n    gold_titles: Sequence[str],\\n    retrieved_titles: Sequence[str],\\n    ks: Sequence[int] = (2, 5, 10),\\n) -> Dict[str, float]:\\n    \\"\\"\\"For each k: |gold ∩ top-k retrieved| / |gold|.\\"\\"\\"\\n    gold = set(gold_titles)\\n    out: Dict[str, float] = {}\\n    for k in ks:\\n        if not gold:\\n            out[f\\"recall@{k}\\"] = 0.0\\n            continue\\n        topk = set(retrieved_titles[:k])\\n        out[f\\"recall@{k}\\"] = len(gold & topk) / len(gold)\\n    return out\\n\\n\\ndef _main() -> None:\\n    assert normalize_answer(\\"The  A.B.C.!\\") == \\"abc\\"\\n    assert em_score([\\"20 March 851\\"], \\"20 march 851.\\") == 1.0\\n    assert em_score([\\"Paris\\"], \\"London\\") == 0.0\\n    assert abs(f1_score([\\"barack obama\\"], \\"obama\\") - (2 * 1.0 * 0.5 / 1.5)) < 1e-9\\n    assert f1_score([\\"cat dog\\"], \\"cat dog\\") == 1.0\\n    r = recall_at_k([\\"A\\", \\"B\\"], [\\"A\\", \\"X\\", \\"B\\", \\"Y\\"], ks=(1, 2, 3))\\n    assert r == {\\"recall@1\\": 0.5, \\"recall@2\\": 0.5, \\"recall@3\\": 1.0}, r\\n    assert recall_at_k([], [\\"A\\"], ks=(2,)) == {\\"recall@2\\": 0.0}\\n    print(\\"evaluation sanity OK\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    _main()\\n", "benchmark/llm_wrap.py": "\\"\\"\\"LLM client wrapper for the gpt-oss-20b Koboldcpp backend.\\n\\nTwo things the reused ``LLMClient`` does not do for us:\\n\\n1. ``strip_reasoning`` is a declared but unimplemented ``POCConfig`` flag. gpt-oss\\n   emits a Harmony \\"analysis\\" channel that, depending on the Koboldcpp build, can\\n   leak into ``message.content`` and break JSON parsing / ``Answer:`` extraction.\\n   ``BenchLLMClient`` strips it AFTER ``super().infer()`` so the SQLite cache keeps\\n   the raw response (still reusable) while callers get clean text.\\n\\n2. A fail-fast health check so a run aborts immediately with a launch hint when\\n   Koboldcpp is not up, instead of after a 10-minute request timeout.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nimport urllib.error\\nimport urllib.request\\n\\nfrom . import _bootstrap  # noqa: F401\\nfrom proprag_poc.config import POCConfig\\nfrom proprag_poc.llm.client import LLMClient\\n\\nlogger = logging.getLogger(__name__)\\n\\n_FINAL_MARKER = \\"<|channel|>final<|message|>\\"\\n_TERMINATORS = (\\"<|end|>\\", \\"<|return|>\\", \\"<|start|>\\")\\n\\n\\ndef strip_gpt_oss_reasoning(content: str) -> str:\\n    \\"\\"\\"Remove the gpt-oss Harmony reasoning channel, keep the final answer text.\\"\\"\\"\\n    if not content:\\n        return content\\n    if _FINAL_MARKER in content:\\n        content = content.split(_FINAL_MARKER)[-1]\\n        for term in _TERMINATORS:\\n            content = content.split(term)[0]\\n        return content.strip()\\n    stripped = content.lstrip()\\n    if stripped.startswith(\\"analysis\\") and \\"assistantfinal\\" in stripped:\\n        return stripped.split(\\"assistantfinal\\")[-1].strip()\\n    return content\\n\\n\\nclass BenchLLMClient(LLMClient):\\n    \\"\\"\\"``LLMClient`` that post-processes gpt-oss reasoning out of every response.\\n\\n    Also disables the OpenAI ``response_format={\\"type\\":\\"json_object\\"}`` constraint:\\n    gpt-oss on Koboldcpp collapses to degenerate output (e.g. an empty ``[]``) when\\n    JSON grammar-constrained, because its Harmony reasoning channels cannot be\\n    expressed under that grammar. We instead rely on the \\"respond ONLY with JSON\\"\\n    prompts plus reasoning-strip and lenient/regex parsing downstream. ``json_mode``\\n    still flows through as before, but no ``response_format`` is sent to the server.\\n    \\"\\"\\"\\n\\n    def __init__(self, config: POCConfig):\\n        super().__init__(config)\\n        self._json_response_format_ok = False\\n\\n    def infer(self, *args, **kwargs):\\n        content, meta, cache_hit = super().infer(*args, **kwargs)\\n        if self.config.strip_reasoning:\\n            content = strip_gpt_oss_reasoning(content)\\n        return content, meta, cache_hit\\n\\n\\ndef check_backend(poc_cfg: POCConfig, timeout: float = 5.0) -> None:\\n    \\"\\"\\"Ping ``{base_url}/models``; raise a launch hint on failure.\\"\\"\\"\\n    base = (poc_cfg.llm_base_url or \\"\\").rstrip(\\"/\\")\\n    url = f\\"{base}/models\\"\\n    hint = (\\n        \\"Cannot reach the LLM backend at \\"\\n        f\\"{poc_cfg.llm_base_url!r}. Start Koboldcpp first:\\\\n\\"\\n        \\"  koboldcpp.exe --model gpt-oss-20b-Q4_K_M.gguf --port 5001 --contextsize 8192\\"\\n    )\\n    try:\\n        with urllib.request.urlopen(url, timeout=timeout) as resp:\\n            if resp.status >= 400:\\n                raise RuntimeError(f\\"{url} returned HTTP {resp.status}\\")\\n    except (urllib.error.URLError, OSError, RuntimeError) as e:\\n        raise RuntimeError(f\\"{hint}\\\\n(underlying error: {e})\\") from e\\n    logger.info(\\"LLM backend reachable at %s\\", poc_cfg.llm_base_url)\\n", "benchmark/proprag_adapter.py": "\\"\\"\\"Index building over the reused ``proprag_poc`` library.\\n\\nThree RAG systems share one chunk embedding store and one embedding model:\\n\\n- BaseRAG cost  = building the shared chunk store (embeddings only).\\n- PropRAG cost  = NER + proposition extraction + knowledge-graph construction.\\n- GraphRAG cost = its own extraction/summaries/reports (see ``graphrag/``).\\n\\nWe deliberately do NOT import ``proprag_poc.core.index`` (it top-imports PyMuPDF /\\ntiktoken via the PDF ingestion path). Instead we replicate the ~50-line\\n``Indexer.build_from_texts`` body directly from ``EmbeddingStore`` + ``Extractor``\\n+ ``GraphBuilder``.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport logging\\nimport os\\nfrom dataclasses import dataclass\\nfrom typing import Dict, List, Tuple\\n\\nimport igraph as ig\\n\\nfrom . import _bootstrap  # noqa: F401\\nfrom .dataset import chunk_text\\nfrom proprag_poc.config import POCConfig\\nfrom proprag_poc.core.baseline_retrievers import BaseRAGRetriever\\nfrom proprag_poc.core.extraction import Extractor\\nfrom proprag_poc.core.graph_builder import GraphBuilder\\nfrom proprag_poc.core.ids import compute_mdhash_id\\nfrom proprag_poc.core.retriever import Retriever\\nfrom proprag_poc.core.store import EmbeddingStore\\nfrom proprag_poc.embedding.encoder import EmbeddingModel\\n\\nlogger = logging.getLogger(__name__)\\n\\n\\n@dataclass\\nclass BenchCorpus:\\n    \\"\\"\\"Duck-typed container satisfying the POC ``Retriever`` + ``BaseRAGRetriever``.\\"\\"\\"\\n\\n    corpus_id: str\\n    chunk_store: EmbeddingStore\\n    entity_store: EmbeddingStore\\n    proposition_store: EmbeddingStore\\n    graph: ig.Graph\\n    proposition_to_entities_map: Dict[str, List[str]]\\n    chunk_propositions: Dict[str, List[Dict]]\\n    chunk_id_to_title: Dict[str, str]\\n\\n\\n# ------------------------------------------------------------------- paths\\ndef corpus_dir(poc_cfg: POCConfig, corpus_id: str) -> str:\\n    return os.path.join(poc_cfg.data_dir, \\"corpora\\", corpus_id)\\n\\n\\ndef _title_map_path(cdir: str) -> str:\\n    return os.path.join(cdir, \\"title_map.json\\")\\n\\n\\ndef _graph_path(cdir: str) -> str:\\n    return os.path.join(cdir, \\"graph.graphml\\")\\n\\n\\ndef _maps_path(cdir: str) -> str:\\n    return os.path.join(cdir, \\"maps.json\\")\\n\\n\\n# --------------------------------------------------------- shared chunk store\\ndef build_base_index(\\n    poc_cfg: POCConfig, corpus_id: str, docs: List[Tuple[str, str]], emb: EmbeddingModel\\n) -> Tuple[EmbeddingStore, Dict[str, str]]:\\n    \\"\\"\\"Build (or load) the shared chunk store. This is the BaseRAG index cost.\\n\\n    Returns the chunk store and a persisted ``chunk_id -> title`` map used for\\n    Recall@k scoring.\\n    \\"\\"\\"\\n    cdir = corpus_dir(poc_cfg, corpus_id)\\n    os.makedirs(cdir, exist_ok=True)\\n\\n    chunk_store = EmbeddingStore(emb, cdir, \\"chunk\\")\\n    texts = [chunk_text(title, text) for title, text in docs]\\n    logger.info(\\"BaseRAG index: embedding %d chunks\\", len(texts))\\n    chunk_store.insert_strings(texts)\\n\\n    chunk_id_to_title = {\\n        compute_mdhash_id(chunk_text(title, text), prefix=\\"chunk-\\"): title\\n        for title, text in docs\\n    }\\n    with open(_title_map_path(cdir), \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(chunk_id_to_title, f)\\n\\n    missing = [cid for cid in chunk_store.get_all_ids() if cid not in chunk_id_to_title]\\n    if missing:\\n        raise RuntimeError(f\\"{len(missing)} chunk ids lack a title mapping\\")\\n    return chunk_store, chunk_id_to_title\\n\\n\\ndef load_title_map(cdir: str) -> Dict[str, str]:\\n    with open(_title_map_path(cdir), \\"r\\", encoding=\\"utf-8\\") as f:\\n        return json.load(f)\\n\\n\\n# --------------------------------------------------------------- PropRAG index\\ndef proprag_exists(poc_cfg: POCConfig, corpus_id: str) -> bool:\\n    cdir = corpus_dir(poc_cfg, corpus_id)\\n    return os.path.isfile(_graph_path(cdir)) and os.path.isfile(_maps_path(cdir))\\n\\n\\ndef build_or_load_proprag(\\n    poc_cfg: POCConfig,\\n    corpus_id: str,\\n    chunk_store: EmbeddingStore,\\n    chunk_id_to_title: Dict[str, str],\\n    emb: EmbeddingModel,\\n    llm,\\n    force: bool = False,\\n) -> BenchCorpus:\\n    \\"\\"\\"Extraction + graph over the shared chunk store (PropRAG index cost).\\"\\"\\"\\n    cdir = corpus_dir(poc_cfg, corpus_id)\\n    entity_store = EmbeddingStore(emb, cdir, \\"entity\\")\\n    proposition_store = EmbeddingStore(emb, cdir, \\"proposition\\")\\n\\n    if proprag_exists(poc_cfg, corpus_id) and not force:\\n        logger.info(\\"Loading existing PropRAG index for %s\\", corpus_id)\\n        graph = ig.Graph.Read_GraphML(_graph_path(cdir))\\n        with open(_maps_path(cdir), \\"r\\", encoding=\\"utf-8\\") as f:\\n            maps = json.load(f)\\n        return BenchCorpus(\\n            corpus_id, chunk_store, entity_store, proposition_store, graph,\\n            maps[\\"proposition_to_entities_map\\"], maps[\\"chunk_propositions\\"],\\n            chunk_id_to_title,\\n        )\\n\\n    # --- replicate Indexer.build_from_texts (chunk store already built) ---\\n    chunk_id_to_text = {\\n        cid: row[\\"content\\"] for cid, row in chunk_store.get_text_for_all_rows().items()\\n    }\\n    logger.info(\\"PropRAG index: extracting NER + propositions for %d chunks\\",\\n                len(chunk_id_to_text))\\n    chunk_propositions = Extractor(llm).batch_extract(chunk_id_to_text)\\n\\n    entities, prop_texts = set(), []\\n    prop_to_entities: Dict[str, List[str]] = {}\\n    for chunk_id, props in chunk_propositions.items():\\n        for prop in props:\\n            entities.update(prop[\\"entities\\"])\\n            pkey = compute_mdhash_id(prop[\\"text\\"], prefix=\\"proposition-\\")\\n            prop_texts.append(prop[\\"text\\"])\\n            prop_to_entities[pkey] = prop[\\"entities\\"]\\n\\n    logger.info(\\"PropRAG index: embedding %d entities, %d propositions\\",\\n                len(entities), len(prop_texts))\\n    entity_store.insert_strings(sorted(entities))\\n    proposition_store.insert_strings(prop_texts)\\n\\n    logger.info(\\"PropRAG index: building knowledge graph\\")\\n    graph = GraphBuilder(poc_cfg).build(\\n        chunk_store.get_all_ids(), chunk_propositions, entity_store, chunk_store\\n    )\\n    graph.write_graphml(_graph_path(cdir))\\n    with open(_maps_path(cdir), \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(\\n            {\\n                \\"proposition_to_entities_map\\": prop_to_entities,\\n                \\"chunk_propositions\\": chunk_propositions,\\n            },\\n            f,\\n        )\\n    logger.info(\\"PropRAG index complete: %d nodes, %d edges\\",\\n                graph.vcount(), graph.ecount())\\n    return BenchCorpus(\\n        corpus_id, chunk_store, entity_store, proposition_store, graph,\\n        prop_to_entities, chunk_propositions, chunk_id_to_title,\\n    )\\n\\n\\n# ----------------------------------------------------------------- retrievers\\ndef make_proprag_retriever(corpus: BenchCorpus, emb: EmbeddingModel, poc_cfg: POCConfig):\\n    return Retriever(corpus, emb, poc_cfg)\\n\\n\\ndef make_baserag_retriever(corpus: BenchCorpus, emb: EmbeddingModel, poc_cfg: POCConfig):\\n    return BaseRAGRetriever(corpus, emb, poc_cfg)\\n", "benchmark/qa.py": "\\"\\"\\"Shared QA prompt + answer extraction for all three systems.\\n\\nOne identical prompt keeps the generation step fair; only the retrieved context\\ndiffers per system. Tuned for 2wiki short spans: brief reasoning, then a final\\n``Answer:`` line carrying the shortest span, which we parse back out for EM/F1.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Dict, List, Tuple\\n\\n_QA_SYSTEM = (\\n    \\"You answer questions using ONLY the provided context passages. Reason \\"\\n    \\"briefly, then finish with a line formatted exactly as \'Answer: <answer>\' \\"\\n    \\"where <answer> is the shortest possible span (a name, date, or short \\"\\n    \\"phrase). If the answer is not in the context, write \'Answer: unknown\'.\\"\\n)\\n\\n\\ndef qa_messages(question: str, context_texts: List[str]) -> List[Dict[str, str]]:\\n    context = \\"\\\\n\\\\n\\".join(f\\"[Passage {i + 1}]\\\\n{p}\\" for i, p in enumerate(context_texts))\\n    return [\\n        {\\"role\\": \\"system\\", \\"content\\": _QA_SYSTEM},\\n        {\\n            \\"role\\": \\"user\\",\\n            \\"content\\": f\\"Context:\\\\n{context}\\\\n\\\\nQuestion: {question}\\\\nReasoning:\\",\\n        },\\n    ]\\n\\n\\ndef extract_answer(raw: str) -> str:\\n    \\"\\"\\"Pull the span after the last ``Answer:`` marker; first line only.\\"\\"\\"\\n    if not raw:\\n        return \\"unknown\\"\\n    tail = raw.rsplit(\\"Answer:\\", 1)[1] if \\"Answer:\\" in raw else raw\\n    first_line = next((ln for ln in tail.strip().splitlines() if ln.strip()), \\"\\")\\n    return first_line.strip().strip(\'\\"\').strip(\\"\'\\").strip(\\". \\").strip() or \\"unknown\\"\\n\\n\\ndef answer_question(llm, question: str, context_texts: List[str], cfg) -> Tuple[str, str]:\\n    \\"\\"\\"Return ``(short_answer, raw_response)``.\\"\\"\\"\\n    raw, _, _ = llm.infer(\\n        qa_messages(question, context_texts),\\n        json_mode=False,\\n        max_completion_tokens=cfg.qa_max_answer_tokens,\\n    )\\n    return extract_answer(raw), raw\\n", "benchmark/report.py": "\\"\\"\\"Aggregate results.jsonl + index_usage.json into metrics.json + report.md.\\n\\nCharts are optional: a guarded matplotlib import degrades to \\"no charts\\" if the\\npackage is missing.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport logging\\nimport os\\nfrom collections import defaultdict\\nfrom typing import Dict, List\\n\\nlogger = logging.getLogger(__name__)\\n\\n_SYSTEMS = (\\"BaseRAG\\", \\"GraphRAG\\", \\"PropRAG\\")\\n_RECALL_KS = (2, 5, 10)\\n\\n\\n# ------------------------------------------------------------------- loading\\ndef _load_rows(run_dir: str) -> List[Dict]:\\n    path = os.path.join(run_dir, \\"results.jsonl\\")\\n    latest: Dict = {}\\n    if not os.path.isfile(path):\\n        return []\\n    with open(path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        for line in f:\\n            line = line.strip()\\n            if not line:\\n                continue\\n            try:\\n                row = json.loads(line)\\n            except json.JSONDecodeError:\\n                continue\\n            if row.get(\\"error\\") or \\"qid\\" not in row or \\"system\\" not in row:\\n                continue\\n            latest[(row[\\"qid\\"], row[\\"system\\"])] = row  # last wins (dedupe)\\n    return list(latest.values())\\n\\n\\ndef _load_json(path: str) -> Dict:\\n    if os.path.isfile(path):\\n        with open(path, \\"r\\", encoding=\\"utf-8\\") as f:\\n            return json.load(f)\\n    return {}\\n\\n\\ndef _mean(xs: List[float]) -> float:\\n    return sum(xs) / len(xs) if xs else 0.0\\n\\n\\n# --------------------------------------------------------------- aggregation\\ndef _aggregate(rows: List[Dict]) -> Dict:\\n    by_system: Dict[str, List[Dict]] = defaultdict(list)\\n    for r in rows:\\n        by_system[r[\\"system\\"]].append(r)\\n\\n    metrics: Dict[str, Dict] = {}\\n    for system, srows in by_system.items():\\n        em = _mean([r.get(\\"em\\", 0.0) for r in srows])\\n        f1 = _mean([r.get(\\"f1\\", 0.0) for r in srows])\\n        recalls = {\\n            f\\"recall@{k}\\": _mean([r.get(\\"recall\\", {}).get(f\\"recall@{k}\\", 0.0) for r in srows])\\n            for k in _RECALL_KS\\n        }\\n        usages = [r.get(\\"usage\\", {}) for r in srows]\\n        metrics[system] = {\\n            \\"n\\": len(srows),\\n            \\"em\\": em,\\n            \\"f1\\": f1,\\n            **recalls,\\n            \\"mean_retrieval_latency_s\\": _mean([r.get(\\"retrieval_latency_s\\", 0.0) for r in srows]),\\n            \\"mean_qa_latency_s\\": _mean([r.get(\\"qa_latency_s\\", 0.0) for r in srows]),\\n            \\"mean_query_prompt_tokens\\": _mean([u.get(\\"chat_prompt_tokens\\", 0) for u in usages]),\\n            \\"mean_query_completion_tokens\\": _mean([u.get(\\"chat_completion_tokens\\", 0) for u in usages]),\\n            \\"mean_chat_calls_per_q\\": _mean([u.get(\\"chat_calls\\", 0) for u in usages]),\\n            \\"by_type\\": _by_type(srows),\\n        }\\n    return metrics\\n\\n\\ndef _by_type(srows: List[Dict]) -> Dict[str, Dict]:\\n    by_type: Dict[str, List[Dict]] = defaultdict(list)\\n    for r in srows:\\n        by_type[r.get(\\"qtype\\", \\"?\\")].append(r)\\n    return {\\n        t: {\\n            \\"n\\": len(rs),\\n            \\"em\\": _mean([r.get(\\"em\\", 0.0) for r in rs]),\\n            \\"f1\\": _mean([r.get(\\"f1\\", 0.0) for r in rs]),\\n            \\"recall@5\\": _mean([r.get(\\"recall\\", {}).get(\\"recall@5\\", 0.0) for r in rs]),\\n        }\\n        for t, rs in sorted(by_type.items())\\n    }\\n\\n\\n# ------------------------------------------------------------------- markdown\\ndef _fmt(x: float, pct: bool = False) -> str:\\n    return f\\"{x * 100:.1f}%\\" if pct else f\\"{x:.1f}\\"\\n\\n\\ndef _render_md(metrics: Dict, index_usage: Dict, manifest: Dict, meta: Dict) -> str:\\n    systems = [s for s in _SYSTEMS if s in metrics] or list(metrics)\\n    lines: List[str] = [\\"# PropRAG vs GraphRAG vs BaseRAG — 2WikiMultiHopQA\\", \\"\\"]\\n    lines.append(f\\"Questions: {manifest.get(\'n_questions\', \'?\')} \\"\\n                 f\\"(seed {manifest.get(\'seed\', \'?\')}, subset {manifest.get(\'subset_hash\', \'?\')[:10]}); \\"\\n                 f\\"corpus {manifest.get(\'corpus_size\', \'?\')} docs.\\")\\n    lines.append(\\"\\")\\n\\n    # Answer + retrieval quality.\\n    lines.append(\\"## Answer & retrieval quality\\")\\n    lines.append(\\"\\")\\n    header = \\"| System | EM | F1 | R@2 | R@5 | R@10 |\\"\\n    lines += [header, \\"|---|---|---|---|---|---|\\"]\\n    for s in systems:\\n        m = metrics[s]\\n        lines.append(\\n            f\\"| {s} | {_fmt(m[\'em\'], True)} | {_fmt(m[\'f1\'], True)} | \\"\\n            f\\"{_fmt(m[\'recall@2\'], True)} | {_fmt(m[\'recall@5\'], True)} | {_fmt(m[\'recall@10\'], True)} |\\"\\n        )\\n    lines.append(\\"\\")\\n\\n    # Per-question-type.\\n    lines.append(\\"## By question type (EM / F1 / R@5)\\")\\n    lines.append(\\"\\")\\n    lines += [\\"| Type | \\" + \\" | \\".join(systems) + \\" |\\", \\"|---|\\" + \\"---|\\" * len(systems)]\\n    all_types = sorted({t for s in systems for t in metrics[s][\\"by_type\\"]})\\n    for t in all_types:\\n        cells = []\\n        for s in systems:\\n            bt = metrics[s][\\"by_type\\"].get(t)\\n            cells.append(\\n                f\\"{_fmt(bt[\'em\'], True)} / {_fmt(bt[\'f1\'], True)} / {_fmt(bt[\'recall@5\'], True)}\\"\\n                if bt else \\"-\\"\\n            )\\n        lines.append(f\\"| {t} | \\" + \\" | \\".join(cells) + \\" |\\")\\n    lines.append(\\"\\")\\n\\n    # Query cost.\\n    lines.append(\\"## Query cost (per question)\\")\\n    lines.append(\\"\\")\\n    lines += [\\"| System | chat calls | prompt tok | completion tok | retr latency (s) | QA latency (s) |\\",\\n              \\"|---|---|---|---|---|---|\\"]\\n    for s in systems:\\n        m = metrics[s]\\n        lines.append(\\n            f\\"| {s} | {m[\'mean_chat_calls_per_q\']:.2f} | {m[\'mean_query_prompt_tokens\']:.0f} | \\"\\n            f\\"{m[\'mean_query_completion_tokens\']:.0f} | {m[\'mean_retrieval_latency_s\']:.2f} | \\"\\n            f\\"{m[\'mean_qa_latency_s\']:.2f} |\\"\\n        )\\n    lines.append(\\"\\")\\n\\n    # Index cost.\\n    lines.append(\\"## Index cost (one-time)\\")\\n    lines.append(\\"\\")\\n    lines += [\\"| System | chat calls | prompt tok | completion tok | embed texts | wall (s) | parse fails |\\",\\n              \\"|---|---|---|---|---|---|---|\\"]\\n    for s in systems:\\n        u = index_usage.get(s, {})\\n        lines.append(\\n            f\\"| {s} | {u.get(\'chat_calls\', 0):.0f} | {u.get(\'chat_prompt_tokens\', 0):.0f} | \\"\\n            f\\"{u.get(\'chat_completion_tokens\', 0):.0f} | {u.get(\'embed_texts\', 0):.0f} | \\"\\n            f\\"{u.get(\'wall_time_s\', 0):.1f} | {u.get(\'parse_failures\', 0)} |\\"\\n        )\\n    lines.append(\\"\\")\\n\\n    # Fairness footer.\\n    lines.append(\\"## Fairness & configuration\\")\\n    lines.append(\\"\\")\\n    lines.append(f\\"- Shared LLM: `{meta.get(\'llm_model\', \'?\')}` @ `{meta.get(\'llm_backend\', \'?\')}`; \\"\\n                 f\\"shared embedder: `{meta.get(\'embedding_model\', \'?\')}`.\\")\\n    lines.append(f\\"- Shared chunk embeddings; qa_top_k={meta.get(\'qa_top_k\', \'?\')}, \\"\\n                 f\\"retrieval_top_k={meta.get(\'retrieval_top_k\', \'?\')}, seed={manifest.get(\'seed\', \'?\')}.\\")\\n    lines.append(f\\"- GraphRAG gleanings={meta.get(\'gr_max_gleanings\', \'?\')} \\"\\n                 \\"(Microsoft default is 1; 0 lowers index cost).\\")\\n    lines.append(\\"- Recall@k uses retrieved documents; GraphRAG QA additionally uses community \\"\\n                 \\"reports + relationships via local-search context assembly.\\")\\n    total_wall = sum(index_usage.get(s, {}).get(\\"wall_time_s\\", 0) for s in systems)\\n    lines.append(f\\"- Total index wall time: {total_wall:.1f}s.\\")\\n    lines.append(\\"\\")\\n    return \\"\\\\n\\".join(lines)\\n\\n\\n# --------------------------------------------------------------------- charts\\ndef _charts(metrics: Dict, index_usage: Dict, run_dir: str) -> None:\\n    try:\\n        import matplotlib\\n        matplotlib.use(\\"Agg\\")\\n        import matplotlib.pyplot as plt\\n    except Exception:  # noqa: BLE001\\n        logger.info(\\"matplotlib unavailable; skipping charts\\")\\n        return\\n\\n    systems = [s for s in _SYSTEMS if s in metrics]\\n    if not systems:\\n        return\\n    charts_dir = os.path.join(run_dir, \\"charts\\")\\n    os.makedirs(charts_dir, exist_ok=True)\\n\\n    # Quality bars.\\n    fig, ax = plt.subplots(figsize=(7, 4))\\n    import numpy as np\\n    x = np.arange(len(systems))\\n    width = 0.25\\n    for i, key in enumerate((\\"em\\", \\"f1\\", \\"recall@5\\")):\\n        ax.bar(x + (i - 1) * width, [metrics[s][key] for s in systems], width, label=key.upper())\\n    ax.set_xticks(x); ax.set_xticklabels(systems); ax.set_ylabel(\\"score\\"); ax.legend()\\n    ax.set_title(\\"Answer quality & Recall@5\\")\\n    fig.tight_layout(); fig.savefig(os.path.join(charts_dir, \\"quality.png\\"), dpi=120); plt.close(fig)\\n\\n    # Index tokens.\\n    fig, ax = plt.subplots(figsize=(7, 4))\\n    prompt = [index_usage.get(s, {}).get(\\"chat_prompt_tokens\\", 0) for s in systems]\\n    compl = [index_usage.get(s, {}).get(\\"chat_completion_tokens\\", 0) for s in systems]\\n    ax.bar(x, prompt, width * 2, label=\\"prompt\\")\\n    ax.bar(x, compl, width * 2, bottom=prompt, label=\\"completion\\")\\n    ax.set_xticks(x); ax.set_xticklabels(systems); ax.set_ylabel(\\"tokens\\"); ax.legend()\\n    ax.set_title(\\"Index chat tokens per system\\")\\n    fig.tight_layout(); fig.savefig(os.path.join(charts_dir, \\"index_tokens.png\\"), dpi=120); plt.close(fig)\\n    logger.info(\\"charts written to %s\\", charts_dir)\\n\\n\\n# ---------------------------------------------------------------------- build\\ndef build(run_dir: str, make_charts: bool = True) -> Dict:\\n    rows = _load_rows(run_dir)\\n    index_usage = _load_json(os.path.join(run_dir, \\"index_usage.json\\"))\\n    manifest = _load_json(os.path.join(run_dir, \\"manifest.json\\"))\\n    meta = _load_json(os.path.join(run_dir, \\"run_meta.json\\"))\\n\\n    metrics = _aggregate(rows)\\n    with open(os.path.join(run_dir, \\"metrics.json\\"), \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(metrics, f, indent=2)\\n\\n    md = _render_md(metrics, index_usage, manifest, meta)\\n    with open(os.path.join(run_dir, \\"report.md\\"), \\"w\\", encoding=\\"utf-8\\") as f:\\n        f.write(md)\\n\\n    if make_charts:\\n        _charts(metrics, index_usage, run_dir)\\n    logger.info(\\"report written to %s\\", os.path.join(run_dir, \\"report.md\\"))\\n    return metrics\\n", "benchmark/results.py": "\\"\\"\\"Append-only, resume-safe results store (one JSON line per (question, system)).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nfrom typing import Dict, Set, Tuple\\n\\n\\nclass ResultsStore:\\n    def __init__(self, run_dir: str):\\n        os.makedirs(run_dir, exist_ok=True)\\n        self.run_dir = run_dir\\n        self.path = os.path.join(run_dir, \\"results.jsonl\\")\\n\\n    def done_keys(self) -> Set[Tuple[str, str]]:\\n        \\"\\"\\"(qid, system) pairs already recorded *successfully*.\\n\\n        Error rows are excluded so a rerun retries transient failures; the report\\n        deduplicates by (qid, system) keeping the last row.\\n        \\"\\"\\"\\n        done: Set[Tuple[str, str]] = set()\\n        if not os.path.isfile(self.path):\\n            return done\\n        with open(self.path, \\"r\\", encoding=\\"utf-8\\") as f:\\n            for line in f:\\n                line = line.strip()\\n                if not line:\\n                    continue\\n                try:\\n                    row = json.loads(line)\\n                except json.JSONDecodeError:\\n                    continue\\n                if row.get(\\"error\\"):\\n                    continue\\n                if \\"qid\\" in row and \\"system\\" in row:\\n                    done.add((row[\\"qid\\"], row[\\"system\\"]))\\n        return done\\n\\n    def append(self, row: Dict) -> None:\\n        with open(self.path, \\"a\\", encoding=\\"utf-8\\") as f:\\n            f.write(json.dumps(row) + \\"\\\\n\\")\\n            f.flush()\\n            os.fsync(f.fileno())\\n", "benchmark/run.py": "\\"\\"\\"Benchmark CLI: index the three systems once over a shared corpus, then run the\\nquestion loop (resume-safe), and build the report.\\n\\n    python -m benchmark.run [--pilot 10] [--questions 50] [--seed 42]\\n                            [--systems BaseRAG,GraphRAG,PropRAG]\\n                            [--force-reindex] [--report-only] [--no-charts]\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport logging\\nimport os\\nimport sys\\nimport time\\nfrom typing import Dict, List\\n\\nfrom . import _bootstrap  # noqa: F401\\nfrom proprag_poc.logging_setup import setup_logging\\nfrom proprag_poc.core.metrics import get_usage_tracker\\nfrom proprag_poc.embedding.encoder import EmbeddingModel\\n\\nfrom . import proprag_adapter as adapter\\nfrom . import report as report_mod\\nfrom .bench_config import BenchmarkConfig\\nfrom .dataset import (\\n    build_corpus, corpus_id, load_questions, pilot_subset, stratified_subset,\\n    subset_hash, write_manifest,\\n)\\nfrom .evaluation import em_score, f1_score, recall_at_k\\nfrom .graphrag import index as gr_index_mod\\nfrom .graphrag.search import GraphRAGLocalRetriever\\nfrom .llm_wrap import BenchLLMClient, check_backend\\nfrom .qa import answer_question\\nfrom .results import ResultsStore\\nfrom .usage import IndexPhase, delta\\n\\nlogger = logging.getLogger(\\"benchmark\\")\\n\\n\\ndef _setup_logging() -> None:\\n    setup_logging()  # configures the proprag_poc tree\\n    handler = logging.StreamHandler(sys.stdout)\\n    handler.setFormatter(\\n        logging.Formatter(\\"%(asctime)s | %(levelname)-5s | %(name)s | %(message)s\\", \\"%H:%M:%S\\")\\n    )\\n    root = logging.getLogger(\\"benchmark\\")\\n    root.setLevel(logging.INFO)\\n    root.handlers.clear()\\n    root.addHandler(handler)\\n    root.propagate = False\\n\\n\\ndef _parse_args(argv=None) -> argparse.Namespace:\\n    p = argparse.ArgumentParser(description=\\"PropRAG vs GraphRAG vs BaseRAG benchmark\\")\\n    p.add_argument(\\"--questions\\", type=int, default=50)\\n    p.add_argument(\\"--pilot\\", type=int, default=None, help=\\"run a stratified k-of-subset pilot\\")\\n    p.add_argument(\\"--seed\\", type=int, default=42)\\n    p.add_argument(\\"--systems\\", type=str, default=\\"BaseRAG,GraphRAG,PropRAG\\")\\n    p.add_argument(\\"--force-reindex\\", action=\\"store_true\\")\\n    p.add_argument(\\"--report-only\\", action=\\"store_true\\")\\n    p.add_argument(\\"--no-charts\\", action=\\"store_true\\")\\n    return p.parse_args(argv)\\n\\n\\ndef _run_id(cfg: BenchmarkConfig) -> str:\\n    rid = f\\"{cfg.n_questions}q_seed{cfg.seed}\\"\\n    return f\\"{rid}_pilot{cfg.pilot}\\" if cfg.pilot else rid\\n\\n\\ndef _guard_meta(run_dir: str, cfg: BenchmarkConfig, poc_cfg, sub_hash: str) -> None:\\n    path = os.path.join(run_dir, \\"run_meta.json\\")\\n    meta = {\\n        \\"subset_hash\\": sub_hash,\\n        \\"seed\\": cfg.seed,\\n        \\"n_questions\\": cfg.n_questions,\\n        \\"pilot\\": cfg.pilot,\\n        \\"llm_backend\\": poc_cfg.llm_backend,\\n        \\"llm_model\\": poc_cfg.llm_model,\\n        \\"embedding_model\\": poc_cfg.embedding_model,\\n        \\"qa_top_k\\": cfg.qa_top_k,\\n        \\"retrieval_top_k\\": cfg.retrieval_top_k,\\n        \\"gr_max_gleanings\\": cfg.gr_max_gleanings,\\n    }\\n    if os.path.isfile(path):\\n        with open(path, \\"r\\", encoding=\\"utf-8\\") as f:\\n            prev = json.load(f)\\n        if prev.get(\\"subset_hash\\") != sub_hash:\\n            raise SystemExit(\\n                f\\"Refusing to resume: existing run at {run_dir} used subset \\"\\n                f\\"{prev.get(\'subset_hash\')}, current is {sub_hash}. Use a new seed/size.\\"\\n            )\\n    with open(path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(meta, f, indent=2)\\n\\n\\ndef _index_all(cfg, poc_cfg, corpus_ident, docs, emb, llm, tracker, run_dir, force) -> Dict:\\n    cdir = adapter.corpus_dir(poc_cfg, corpus_ident)\\n    usage_path = os.path.join(run_dir, \\"index_usage.json\\")\\n    index_usage: Dict[str, Dict] = {}\\n\\n    with IndexPhase(tracker, \\"BaseRAG\\") as phase:\\n        chunk_store, chunk_id_to_title = adapter.build_base_index(poc_cfg, corpus_ident, docs, emb)\\n    index_usage[\\"BaseRAG\\"] = phase.usage\\n\\n    with IndexPhase(tracker, \\"PropRAG\\") as phase:\\n        corpus = adapter.build_or_load_proprag(\\n            poc_cfg, corpus_ident, chunk_store, chunk_id_to_title, emb, llm, force=force\\n        )\\n    index_usage[\\"PropRAG\\"] = phase.usage\\n\\n    with IndexPhase(tracker, \\"GraphRAG\\", extra_scopes=[\\"index::GraphRAG\\"]) as phase:\\n        gr_index = gr_index_mod.build_or_load(\\n            poc_cfg, cfg, cdir, chunk_store, emb, llm, tracker, force=force\\n        )\\n    gr_usage = dict(phase.usage)\\n    gr_usage[\\"parse_failures\\"] = gr_index.n_extract_failures\\n    index_usage[\\"GraphRAG\\"] = gr_usage\\n\\n    with open(usage_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(index_usage, f, indent=2)\\n\\n    return {\\n        \\"corpus\\": corpus,\\n        \\"chunk_id_to_title\\": chunk_id_to_title,\\n        \\"gr_index\\": gr_index,\\n        \\"index_usage\\": index_usage,\\n    }\\n\\n\\ndef _build_retrievers(built, emb, poc_cfg, cfg, systems):\\n    retrievers = {}\\n    if \\"BaseRAG\\" in systems:\\n        retrievers[\\"BaseRAG\\"] = adapter.make_baserag_retriever(built[\\"corpus\\"], emb, poc_cfg)\\n    if \\"PropRAG\\" in systems:\\n        retrievers[\\"PropRAG\\"] = adapter.make_proprag_retriever(built[\\"corpus\\"], emb, poc_cfg)\\n    if \\"GraphRAG\\" in systems:\\n        retrievers[\\"GraphRAG\\"] = GraphRAGLocalRetriever(built[\\"gr_index\\"], emb, poc_cfg, cfg)\\n    return retrievers\\n\\n\\ndef _question_loop(questions, retrievers, systems, cfg, poc_cfg, llm, tracker, store,\\n                   chunk_id_to_title):\\n    done = store.done_keys()\\n    per_system_seconds: Dict[str, List[float]] = {s: [] for s in systems}\\n\\n    for qi, q in enumerate(questions, 1):\\n        logger.info(\\"Q %d/%d [%s] %s\\", qi, len(questions), q.qtype, q.question[:80])\\n        for system in systems:\\n            if (q.qid, system) in done:\\n                continue\\n            retriever = retrievers[system]\\n            scope = f\\"q::{system}\\"\\n            before = tracker.snapshot(scope).as_dict()\\n            try:\\n                t0 = time.monotonic()\\n                with tracker.scope(scope):\\n                    passages = retriever.retrieve(q.question, top_k=poc_cfg.retrieval_top_k)\\n                    retrieval_latency = time.monotonic() - t0\\n\\n                    retrieved_titles = [\\n                        chunk_id_to_title.get(p.chunk_id, \\"\\") for p in passages\\n                    ]\\n                    recall = recall_at_k(q.gold_titles, retrieved_titles, cfg.recall_ks)\\n\\n                    if system == \\"GraphRAG\\":\\n                        context = retriever.build_qa_context(q.question, passages)\\n                    else:\\n                        context = [p.text for p in passages[: cfg.qa_top_k]]\\n\\n                    t1 = time.monotonic()\\n                    answer, raw = answer_question(llm, q.question, context, cfg)\\n                    qa_latency = time.monotonic() - t1\\n\\n                usage = delta(before, tracker.snapshot(scope).as_dict())\\n                row = {\\n                    \\"qid\\": q.qid, \\"qtype\\": q.qtype, \\"system\\": system,\\n                    \\"question\\": q.question, \\"gold_answer\\": q.answer,\\n                    \\"gold_titles\\": q.gold_titles, \\"answer\\": answer, \\"raw_answer\\": raw,\\n                    \\"retrieved\\": [\\n                        {\\"chunk_id\\": p.chunk_id, \\"title\\": chunk_id_to_title.get(p.chunk_id, \\"\\"),\\n                         \\"score\\": p.score} for p in passages\\n                    ],\\n                    \\"recall\\": recall,\\n                    \\"em\\": em_score(q.gold_answers, answer),\\n                    \\"f1\\": f1_score(q.gold_answers, answer),\\n                    \\"retrieval_latency_s\\": round(retrieval_latency, 3),\\n                    \\"qa_latency_s\\": round(qa_latency, 3),\\n                    \\"usage\\": usage,\\n                    \\"ts\\": time.time(),\\n                }\\n                store.append(row)\\n                per_system_seconds[system].append(retrieval_latency + qa_latency)\\n            except Exception as e:  # noqa: BLE001 - one failure must not kill the run\\n                logger.exception(\\"Q %s system %s failed\\", q.qid, system)\\n                store.append({\\n                    \\"qid\\": q.qid, \\"qtype\\": q.qtype, \\"system\\": system, \\"error\\": str(e),\\n                    \\"ts\\": time.time(),\\n                })\\n    return per_system_seconds\\n\\n\\ndef main(argv=None) -> None:\\n    args = _parse_args(argv)\\n    _setup_logging()\\n\\n    project_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\\n    cfg = BenchmarkConfig(\\n        project_dir=project_dir, n_questions=args.questions, seed=args.seed, pilot=args.pilot,\\n    )\\n    poc_cfg = cfg.make_poc_config()\\n    systems = [s.strip() for s in args.systems.split(\\",\\") if s.strip()]\\n\\n    run_dir = os.path.join(cfg.data_dir, \\"benchmark\\", _run_id(cfg))\\n    os.makedirs(run_dir, exist_ok=True)\\n\\n    if args.report_only:\\n        report_mod.build(run_dir, make_charts=not args.no_charts)\\n        return\\n\\n    check_backend(poc_cfg)\\n\\n    all_qs = load_questions(cfg.dataset_path)\\n    subset = stratified_subset(all_qs, cfg.n_questions, cfg.seed)\\n    questions = pilot_subset(subset, cfg.pilot, cfg.seed) if cfg.pilot else subset\\n\\n    titles = build_corpus(questions, cfg.corpus_path)\\n    docs = [(t, titles[t]) for t in sorted(titles)]\\n    sub_hash = subset_hash(questions)\\n    tag = f\\"n{cfg.n_questions}\\" + (f\\"_pilot{cfg.pilot}\\" if cfg.pilot else \\"\\")\\n    corpus_ident = corpus_id(questions, tag)\\n\\n    write_manifest(run_dir, questions, titles, cfg.seed, cfg)\\n    _guard_meta(run_dir, cfg, poc_cfg, sub_hash)\\n\\n    emb = EmbeddingModel(poc_cfg)\\n    llm = BenchLLMClient(poc_cfg)\\n    tracker = get_usage_tracker()\\n\\n    logger.info(\\"Indexing %d systems over %d docs (corpus %s)\\", len(systems), len(docs), corpus_ident)\\n    built = _index_all(cfg, poc_cfg, corpus_ident, docs, emb, llm, tracker, run_dir, args.force_reindex)\\n\\n    retrievers = _build_retrievers(built, emb, poc_cfg, cfg, systems)\\n\\n    store = ResultsStore(run_dir)\\n    t_start = time.monotonic()\\n    per_system_seconds = _question_loop(\\n        questions, retrievers, systems, cfg, poc_cfg, llm, tracker, store,\\n        built[\\"chunk_id_to_title\\"],\\n    )\\n    wall = time.monotonic() - t_start\\n\\n    report_mod.build(run_dir, make_charts=not args.no_charts)\\n    _print_summary(run_dir, cfg, per_system_seconds, wall)\\n\\n\\ndef _print_summary(run_dir, cfg, per_system_seconds, wall) -> None:\\n    print(f\\"\\\\nReport: {os.path.join(run_dir, \'report.md\')}\\")\\n    if cfg.pilot:\\n        newly = [s for secs in per_system_seconds.values() for s in secs]\\n        if newly:\\n            mean_q = sum(newly) / len(newly)\\n            n_systems = len(per_system_seconds)\\n            projected = mean_q * n_systems * cfg.n_questions\\n            print(f\\"Pilot: mean {mean_q:.1f}s per (system,question); \\"\\n                  f\\"projected full-run query wall ~ {projected / 60:.0f} min \\"\\n                  f\\"(indexing excluded, cached calls free).\\")\\n    print(f\\"Query-loop wall this run: {wall / 60:.1f} min\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "benchmark/smoke.py": "\\"\\"\\"End-to-end smoke test on 4 tiny docs + 2 questions.\\n\\nThe Koboldcpp-up verification gate: exercises all three indexes, the query loop\\nand the report in a handful of LLM calls, and asserts the failure modes we care\\nabout most (JSON parse, reasoning leakage, Answer: extraction).\\n\\n    python -m benchmark.smoke\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport logging\\nimport os\\n\\nfrom . import _bootstrap  # noqa: F401\\nfrom proprag_poc.core.metrics import get_usage_tracker\\nfrom proprag_poc.embedding.encoder import EmbeddingModel\\n\\nfrom .bench_config import BenchmarkConfig\\nfrom .dataset import BenchQuestion\\nfrom .llm_wrap import BenchLLMClient, check_backend\\nfrom .report import build as build_report\\nfrom .results import ResultsStore\\nfrom .run import _build_retrievers, _index_all, _question_loop, _setup_logging\\n\\nlogger = logging.getLogger(\\"benchmark\\")\\n\\n_DOCS = [\\n    (\\"Ada Lovelace\\", \\"Ada Lovelace was an English mathematician known for her work on \\"\\n                     \\"Charles Babbage\'s Analytical Engine. She is regarded as the first \\"\\n                     \\"computer programmer.\\"),\\n    (\\"Charles Babbage\\", \\"Charles Babbage was an English mathematician who originated the \\"\\n                        \\"concept of a programmable computer, the Analytical Engine.\\"),\\n    (\\"Analytical Engine\\", \\"The Analytical Engine was a proposed mechanical general-purpose \\"\\n                         \\"computer designed by Charles Babbage in the 1830s.\\"),\\n    (\\"Alan Turing\\", \\"Alan Turing was an English mathematician who formalized the concepts of \\"\\n                   \\"algorithm and computation with the Turing machine.\\"),\\n]\\n\\n_QUESTIONS = [\\n    BenchQuestion(\\n        qid=\\"smoke-1\\", qtype=\\"compositional\\",\\n        question=\\"Who designed the machine that Ada Lovelace is known for working on?\\",\\n        answer=\\"Charles Babbage\\",\\n        gold_titles=[\\"Ada Lovelace\\", \\"Analytical Engine\\", \\"Charles Babbage\\"],\\n        context_titles=[\\"Ada Lovelace\\", \\"Analytical Engine\\", \\"Charles Babbage\\"],\\n    ),\\n    BenchQuestion(\\n        qid=\\"smoke-2\\", qtype=\\"comparison\\",\\n        question=\\"Were Ada Lovelace and Alan Turing both English mathematicians?\\",\\n        answer=\\"yes\\",\\n        gold_titles=[\\"Ada Lovelace\\", \\"Alan Turing\\"],\\n        context_titles=[\\"Ada Lovelace\\", \\"Alan Turing\\"],\\n    ),\\n]\\n\\n\\ndef main() -> None:\\n    _setup_logging()\\n    project_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\\n    cfg = BenchmarkConfig(project_dir=project_dir, n_questions=len(_QUESTIONS), seed=42)\\n    poc_cfg = cfg.make_poc_config()\\n    check_backend(poc_cfg)\\n\\n    emb = EmbeddingModel(poc_cfg)\\n    llm = BenchLLMClient(poc_cfg)\\n    tracker = get_usage_tracker()\\n\\n    run_dir = os.path.join(cfg.data_dir, \\"benchmark\\", \\"smoke\\")\\n    os.makedirs(run_dir, exist_ok=True)\\n    systems = [\\"BaseRAG\\", \\"GraphRAG\\", \\"PropRAG\\"]\\n    corpus_ident = \\"2wiki_smoke\\"\\n\\n    built = _index_all(cfg, poc_cfg, corpus_ident, _DOCS, emb, llm, tracker, run_dir, force=True)\\n\\n    assert built[\\"gr_index\\"].entities, \\"GraphRAG extraction produced no entities\\"\\n    assert len(built[\\"index_usage\\"]) == 3, \\"expected 3 index_usage entries\\"\\n\\n    retrievers = _build_retrievers(built, emb, poc_cfg, cfg, systems)\\n    store = ResultsStore(run_dir)\\n    # Fresh results for a clean assertion.\\n    if os.path.isfile(store.path):\\n        os.remove(store.path)\\n    _question_loop(_QUESTIONS, retrievers, systems, cfg, poc_cfg, llm, tracker, store,\\n                   built[\\"chunk_id_to_title\\"])\\n\\n    rows = [json.loads(l) for l in open(store.path, encoding=\\"utf-8\\") if l.strip()]\\n    assert rows, \\"no result rows written\\"\\n    for row in rows:\\n        assert not row.get(\\"error\\"), f\\"row errored: {row.get(\'error\')}\\"\\n        ans = row[\\"answer\\"]\\n        assert ans and ans != \\"unknown\\", f\\"empty/unknown answer for {row[\'qid\']}/{row[\'system\']}\\"\\n        assert \\"<|\\" not in ans and \\"analysis\\" not in ans[:20].lower(), \\\\\\n            f\\"reasoning leaked into answer: {ans!r}\\"\\n\\n    build_report(run_dir, make_charts=False)\\n    assert os.path.isfile(os.path.join(run_dir, \\"report.md\\")), \\"report.md not written\\"\\n    print(\\"SMOKE OK — 3 indexes built, answers parsed, report written:\\",\\n          os.path.join(run_dir, \\"report.md\\"))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "benchmark/usage.py": "\\"\\"\\"Per-phase token/latency attribution over the shared ``UsageTracker``.\\n\\nThe tracker\'s ``scope()`` is thread-local, and the reused ``Extractor.batch_extract``\\nruns in a thread pool, so index-time extraction lands in the default ``\\"index\\"``\\nscope regardless of any active scope. Index builds run sequentially, so each\\nphase\'s cost is the *delta* of the relevant scopes across the phase.\\n\\n- BaseRAG / PropRAG index: their calls land in ``\\"index\\"`` -> delta of ``\\"index\\"``.\\n- GraphRAG index: worker calls set ``\\"index::GraphRAG\\"`` explicitly, main-thread\\n  calls land in ``\\"index\\"``; summing both scopes\' deltas covers both paths.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport time\\nfrom typing import Dict, List\\n\\n\\ndef delta(before: Dict[str, float], after: Dict[str, float]) -> Dict[str, float]:\\n    \\"\\"\\"Field-wise ``after - before`` over ``Snapshot.as_dict()`` outputs.\\"\\"\\"\\n    keys = set(before) | set(after)\\n    return {k: after.get(k, 0) - before.get(k, 0) for k in keys}\\n\\n\\ndef _add(into: Dict[str, float], other: Dict[str, float]) -> None:\\n    for k, v in other.items():\\n        into[k] = into.get(k, 0) + v\\n\\n\\nclass IndexPhase:\\n    \\"\\"\\"Context manager capturing the usage delta of one index phase.\\n\\n    ``self.usage`` is populated on exit: summed deltas of ``\\"index\\"`` plus any\\n    ``extra_scopes`` (e.g. ``\\"index::GraphRAG\\"``), and ``wall_time_s``.\\n    \\"\\"\\"\\n\\n    def __init__(self, tracker, name: str, extra_scopes: List[str] = None):\\n        self._tracker = tracker\\n        self.name = name\\n        self._scopes = [\\"index\\"] + list(extra_scopes or [])\\n        self._before: Dict[str, Dict[str, float]] = {}\\n        self._t0 = 0.0\\n        self.usage: Dict[str, float] = {}\\n\\n    def __enter__(self) -> \\"IndexPhase\\":\\n        self._before = {s: self._tracker.snapshot(s).as_dict() for s in self._scopes}\\n        self._t0 = time.monotonic()\\n        return self\\n\\n    def __exit__(self, *exc) -> bool:\\n        wall = time.monotonic() - self._t0\\n        combined: Dict[str, float] = {}\\n        for s in self._scopes:\\n            _add(combined, delta(self._before[s], self._tracker.snapshot(s).as_dict()))\\n        combined[\\"wall_time_s\\"] = round(wall, 3)\\n        self.usage = combined\\n        return False\\n", "benchmark/graphrag/__init__.py": "\\"\\"\\"Faithful Microsoft-style GraphRAG (own extraction, Leiden communities, reports).\\n\\nBuilt new so its indexing cost is genuinely measured, rather than reusing the\\nPropRAG graph. Local search only.\\n\\"\\"\\"\\n", "benchmark/graphrag/communities.py": "\\"\\"\\"Leiden community detection (igraph built-in, no leidenalg) + LLM reports.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport logging\\nimport re\\nfrom typing import Dict, List\\n\\nimport igraph as ig\\n\\nfrom .extract import EntityRec, RelRec\\nfrom . import prompts\\n\\nlogger = logging.getLogger(__name__)\\n\\n\\ndef build_graph(entities: Dict[str, EntityRec], rels: List[RelRec]) -> ig.Graph:\\n    names = list(entities.keys())\\n    idx = {n: i for i, n in enumerate(names)}\\n    g = ig.Graph()\\n    g.add_vertices(len(names))\\n    g.vs[\\"name\\"] = names\\n\\n    edges, weights = [], []\\n    for rel in rels:\\n        if rel.src_key in idx and rel.dst_key in idx and rel.src_key != rel.dst_key:\\n            edges.append((idx[rel.src_key], idx[rel.dst_key]))\\n            weights.append(float(rel.weight) if rel.weight > 0 else 1.0)\\n    g.add_edges(edges)\\n    g.es[\\"weight\\"] = weights\\n    logger.info(\\"GraphRAG entity graph: %d nodes, %d edges\\", g.vcount(), g.ecount())\\n    return g\\n\\n\\ndef detect_communities(g: ig.Graph) -> Dict[str, int]:\\n    \\"\\"\\"entity key -> community id via weighted-modularity Leiden.\\"\\"\\"\\n    if g.vcount() == 0:\\n        return {}\\n    if g.ecount() == 0:\\n        return {g.vs[i][\\"name\\"]: i for i in range(g.vcount())}\\n    clustering = g.community_leiden(\\n        objective_function=\\"modularity\\", weights=g.es[\\"weight\\"], n_iterations=5\\n    )\\n    membership = clustering.membership\\n    n = len(set(membership))\\n    logger.info(\\"GraphRAG detected %d communities over %d entities\\", n, g.vcount())\\n    return {g.vs[i][\\"name\\"]: membership[i] for i in range(g.vcount())}\\n\\n\\ndef _loads_lenient(raw: str):\\n    s = raw.strip()\\n    s = re.sub(r\\"^```(?:json)?\\", \\"\\", s).strip()\\n    s = re.sub(r\\"```$\\", \\"\\", s).strip()\\n    return json.loads(s)\\n\\n\\ndef community_report(\\n    llm,\\n    member_keys: List[str],\\n    entities: Dict[str, EntityRec],\\n    intra_rels: List[RelRec],\\n    cfg,\\n) -> Dict:\\n    \\"\\"\\"Generate a report for one community; degrade to a minimal report on failure.\\"\\"\\"\\n    ent_lines = []\\n    for k in member_keys:\\n        rec = entities.get(k)\\n        if rec is None:\\n            continue\\n        ent_lines.append(f\\"{rec.name} ({rec.type}): {rec.merged_description()}\\")\\n    entities_block = \\"\\\\n\\".join(ent_lines)\\n\\n    rel_lines = [\\n        f\\"{entities[r.src_key].name} -> {entities[r.dst_key].name}: \\"\\n        f\\"{\' ; \'.join(dict.fromkeys(r.descriptions)) or \'related\'} (w={r.weight:.0f})\\"\\n        for r in sorted(intra_rels, key=lambda x: x.weight, reverse=True)\\n        if r.src_key in entities and r.dst_key in entities\\n    ]\\n    rels_block = \\"\\\\n\\".join(rel_lines)\\n\\n    # Truncate the combined context to the configured character budget.\\n    budget = cfg.gr_report_max_input_chars\\n    if len(entities_block) + len(rels_block) > budget:\\n        entities_block = entities_block[: budget // 2]\\n        rels_block = rels_block[: budget // 2]\\n\\n    try:\\n        raw, _, _ = llm.infer(\\n            prompts.community_report_messages(entities_block, rels_block),\\n            json_mode=True,\\n            max_completion_tokens=cfg.report_max_tokens,\\n        )\\n        report = _loads_lenient(raw)\\n        if not isinstance(report, dict):\\n            raise ValueError(\\"report is not a JSON object\\")\\n        report.setdefault(\\"title\\", f\\"Community of {member_keys[0] if member_keys else \'?\'}\\")\\n        report.setdefault(\\"summary\\", \\"\\")\\n        report.setdefault(\\"rating\\", 0)\\n        report.setdefault(\\"findings\\", [])\\n        return report\\n    except Exception as e:  # noqa: BLE001\\n        logger.warning(\\"community report failed, using stub: %s\\", e)\\n        titles = \\", \\".join(entities[k].name for k in member_keys[:3] if k in entities)\\n        return {\\n            \\"title\\": titles or \\"Community\\",\\n            \\"summary\\": entities_block[:400],\\n            \\"rating\\": 0,\\n            \\"findings\\": [],\\n            \\"parse_failed\\": True,\\n        }\\n", "benchmark/graphrag/extract.py": "\\"\\"\\"Typed entity + relationship extraction with a multi-tier JSON-repair chain.\\n\\nMirrors the repair strategy proven in ``proprag_poc/core/extraction.py``:\\njson_mode -> lenient parse -> ``fix_json`` repair call -> regex object salvage ->\\nempty result with a counted warning. Extraction workers set the\\n``index::GraphRAG`` usage scope so token attribution survives the thread pool.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport logging\\nimport re\\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\\nfrom dataclasses import dataclass, field\\nfrom typing import Dict, List, Set, Tuple\\n\\nfrom proprag_poc.llm import prompts as poc_prompts\\nfrom . import prompts\\n\\nlogger = logging.getLogger(__name__)\\n\\n_ENTITY_OBJ = re.compile(\\n    r\'\\\\{\\\\s*\\"name\\"\\\\s*:\\\\s*\\"(?P<name>(?:\\\\\\\\.|[^\\"\\\\\\\\])*)\\"\\\\s*,\\\\s*\\"type\\"\\\\s*:\\\\s*\'\\n    r\'\\"(?P<type>(?:\\\\\\\\.|[^\\"\\\\\\\\])*)\\"\\\\s*,\\\\s*\\"description\\"\\\\s*:\\\\s*\'\\n    r\'\\"(?P<desc>(?:\\\\\\\\.|[^\\"\\\\\\\\])*)\\"\',\\n    re.DOTALL,\\n)\\n_REL_OBJ = re.compile(\\n    r\'\\\\{\\\\s*\\"source\\"\\\\s*:\\\\s*\\"(?P<src>(?:\\\\\\\\.|[^\\"\\\\\\\\])*)\\"\\\\s*,\\\\s*\\"target\\"\\\\s*:\\\\s*\'\\n    r\'\\"(?P<dst>(?:\\\\\\\\.|[^\\"\\\\\\\\])*)\\"\\\\s*,\\\\s*\\"description\\"\\\\s*:\\\\s*\'\\n    r\'\\"(?P<desc>(?:\\\\\\\\.|[^\\"\\\\\\\\])*)\\"(?:\\\\s*,\\\\s*\\"strength\\"\\\\s*:\\\\s*(?P<strength>\\\\d+))?\',\\n    re.DOTALL,\\n)\\n\\n\\n@dataclass\\nclass EntityRec:\\n    key: str\\n    name: str\\n    type: str\\n    descriptions: List[str] = field(default_factory=list)\\n    chunk_ids: Set[str] = field(default_factory=set)\\n    summary: str = \\"\\"\\n\\n    def merged_description(self) -> str:\\n        return self.summary or \\" \\".join(dict.fromkeys(self.descriptions))\\n\\n\\n@dataclass\\nclass RelRec:\\n    src_key: str\\n    dst_key: str\\n    descriptions: List[str] = field(default_factory=list)\\n    weight: float = 0.0\\n    chunk_ids: Set[str] = field(default_factory=set)\\n\\n\\ndef _loads_lenient(raw: str):\\n    s = raw.strip()\\n    s = re.sub(r\\"^```(?:json)?\\", \\"\\", s).strip()\\n    s = re.sub(r\\"```$\\", \\"\\", s).strip()\\n    return json.loads(s)\\n\\n\\ndef _normalize(obj) -> Dict[str, list]:\\n    entities, rels = [], []\\n    if isinstance(obj, dict):\\n        for e in obj.get(\\"entities\\", []) or []:\\n            if isinstance(e, dict) and e.get(\\"name\\"):\\n                entities.append(e)\\n        for r in obj.get(\\"relationships\\", []) or []:\\n            if isinstance(r, dict) and r.get(\\"source\\") and r.get(\\"target\\"):\\n                rels.append(r)\\n    return {\\"entities\\": entities, \\"relationships\\": rels}\\n\\n\\ndef parse_extraction(raw: str) -> Dict[str, list]:\\n    \\"\\"\\"Lenient JSON parse; fall back to per-object regex salvage.\\"\\"\\"\\n    try:\\n        parsed = _normalize(_loads_lenient(raw))\\n        if parsed[\\"entities\\"] or parsed[\\"relationships\\"]:\\n            return parsed\\n    except Exception:  # noqa: BLE001\\n        pass\\n\\n    entities = [\\n        {\\"name\\": m.group(\\"name\\"), \\"type\\": m.group(\\"type\\"), \\"description\\": m.group(\\"desc\\")}\\n        for m in _ENTITY_OBJ.finditer(raw)\\n    ]\\n    rels = []\\n    for m in _REL_OBJ.finditer(raw):\\n        rels.append({\\n            \\"source\\": m.group(\\"src\\"),\\n            \\"target\\": m.group(\\"dst\\"),\\n            \\"description\\": m.group(\\"desc\\"),\\n            \\"strength\\": int(m.group(\\"strength\\")) if m.group(\\"strength\\") else 5,\\n        })\\n    return {\\"entities\\": entities, \\"relationships\\": rels}\\n\\n\\ndef extract_chunk(llm, text: str, cfg) -> Tuple[Dict[str, list], bool]:\\n    \\"\\"\\"Return (parsed, failed). ``failed`` is True only when nothing was salvaged.\\"\\"\\"\\n    raw, _, _ = llm.infer(\\n        prompts.extraction_messages(text, list(cfg.gr_entity_types)),\\n        json_mode=True,\\n        max_completion_tokens=cfg.extract_max_tokens,\\n    )\\n    parsed = parse_extraction(raw)\\n    if not parsed[\\"entities\\"] and not parsed[\\"relationships\\"]:\\n        fixed, _, _ = llm.infer(poc_prompts.fix_json_messages(raw), json_mode=True)\\n        parsed = parse_extraction(fixed)\\n\\n    failed = not parsed[\\"entities\\"] and not parsed[\\"relationships\\"]\\n\\n    for _ in range(max(0, cfg.gr_max_gleanings)):\\n        prev = json.dumps(parsed)\\n        graw, _, _ = llm.infer(\\n            prompts.gleaning_messages(text, prev, list(cfg.gr_entity_types)),\\n            json_mode=True,\\n            max_completion_tokens=cfg.extract_max_tokens,\\n        )\\n        extra = parse_extraction(graw)\\n        parsed[\\"entities\\"].extend(extra[\\"entities\\"])\\n        parsed[\\"relationships\\"].extend(extra[\\"relationships\\"])\\n\\n    return parsed, failed\\n\\n\\ndef _ekey(name: str) -> str:\\n    return name.strip().lower()\\n\\n\\ndef batch_extract(\\n    llm, chunk_texts: Dict[str, str], cfg, tracker\\n) -> Tuple[Dict[str, EntityRec], List[RelRec], int]:\\n    \\"\\"\\"Parallel extraction + merge. Returns (entities, relationships, n_failures).\\"\\"\\"\\n    keys = list(chunk_texts.keys())\\n    max_workers = llm.config.llm_max_workers\\n    logger.info(\\"GraphRAG extract: %d chunks, %d workers\\", len(keys), max_workers)\\n\\n    def _work(cid: str) -> Tuple[str, Dict[str, list], bool]:\\n        with tracker.scope(\\"index::GraphRAG\\"):\\n            parsed, failed = extract_chunk(llm, chunk_texts[cid], cfg)\\n        return cid, parsed, failed\\n\\n    per_chunk: Dict[str, Dict[str, list]] = {}\\n    n_failures = 0\\n    with ThreadPoolExecutor(max_workers=max_workers) as ex:\\n        futs = {ex.submit(_work, cid): cid for cid in keys}\\n        for done, fut in enumerate(as_completed(futs), 1):\\n            cid, parsed, failed = fut.result()\\n            per_chunk[cid] = parsed\\n            n_failures += int(failed)\\n            logger.info(\\"GraphRAG extract %d/%d chunks done\\", done, len(keys))\\n\\n    entities: Dict[str, EntityRec] = {}\\n    relationships: Dict[Tuple[str, str], RelRec] = {}\\n\\n    def _ensure_entity(name: str, etype: str = \\"other\\") -> EntityRec:\\n        k = _ekey(name)\\n        rec = entities.get(k)\\n        if rec is None:\\n            rec = EntityRec(key=k, name=name.strip(), type=etype or \\"other\\")\\n            entities[k] = rec\\n        elif not rec.type or rec.type == \\"other\\":\\n            rec.type = etype or rec.type\\n        return rec\\n\\n    for cid, parsed in per_chunk.items():\\n        for e in parsed[\\"entities\\"]:\\n            rec = _ensure_entity(e[\\"name\\"], e.get(\\"type\\", \\"other\\"))\\n            desc = (e.get(\\"description\\") or \\"\\").strip()\\n            if desc:\\n                rec.descriptions.append(desc)\\n            rec.chunk_ids.add(cid)\\n        for r in parsed[\\"relationships\\"]:\\n            src = _ensure_entity(r[\\"source\\"])          # stub endpoints (Microsoft behavior)\\n            dst = _ensure_entity(r[\\"target\\"])\\n            if src.key == dst.key:\\n                continue\\n            pair = tuple(sorted((src.key, dst.key)))\\n            rel = relationships.get(pair)\\n            if rel is None:\\n                rel = RelRec(src_key=pair[0], dst_key=pair[1])\\n                relationships[pair] = rel\\n            desc = (r.get(\\"description\\") or \\"\\").strip()\\n            if desc:\\n                rel.descriptions.append(desc)\\n            try:\\n                rel.weight += float(r.get(\\"strength\\", 5) or 5)\\n            except (TypeError, ValueError):\\n                rel.weight += 5.0\\n            rel.chunk_ids.add(cid)\\n\\n    logger.info(\\"GraphRAG merged: %d entities, %d relationships (%d parse failures)\\",\\n                len(entities), len(relationships), n_failures)\\n    return entities, list(relationships.values()), n_failures\\n\\n\\ndef summarize_entities(llm, entities: Dict[str, EntityRec], cfg) -> None:\\n    \\"\\"\\"LLM-merge descriptions only when their joined length exceeds the threshold.\\"\\"\\"\\n    n = 0\\n    for rec in entities.values():\\n        joined = \\" \\".join(dict.fromkeys(rec.descriptions))\\n        if len(rec.descriptions) > 1 and len(joined) > cfg.gr_summarize_desc_threshold:\\n            raw, _, _ = llm.infer(\\n                prompts.summarize_descriptions_messages(rec.name, rec.descriptions),\\n                json_mode=True,\\n                max_completion_tokens=cfg.summarize_max_tokens,\\n            )\\n            try:\\n                rec.summary = (_loads_lenient(raw).get(\\"description\\") or \\"\\").strip()\\n            except Exception:  # noqa: BLE001\\n                rec.summary = joined[: cfg.gr_summarize_desc_threshold]\\n            n += 1\\n    logger.info(\\"GraphRAG summarized %d oversized entity descriptions\\", n)\\n", "benchmark/graphrag/index.py": "\\"\\"\\"GraphRAG index: build / persist / load.\\n\\nReuses the SHARED chunk embedding store (no re-embedding of passages). Persists\\nunder ``data/corpora/<corpus_id>/graphrag/``:\\n  extraction.json   - entities + relationships checkpoint (after batch_extract)\\n  entities.json     - merged/summarized entities (with embed text + provenance)\\n  relationships.json, communities.json, reports.json\\n  graph.graphml     - entity graph\\n  gr_entity_*       - EmbeddingStore for \\"name: description\\" vectors\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport logging\\nimport os\\nfrom dataclasses import dataclass, field\\nfrom typing import Dict, List\\n\\nimport igraph as ig\\nimport numpy as np\\n\\nfrom proprag_poc.core.ids import compute_mdhash_id\\nfrom proprag_poc.core.store import EmbeddingStore\\nfrom proprag_poc.embedding.encoder import EmbeddingModel\\n\\nfrom . import communities as comm\\nfrom .extract import EntityRec, RelRec, batch_extract, summarize_entities\\n\\nlogger = logging.getLogger(__name__)\\n\\n_GR_NAMESPACE = \\"gr_entity\\"\\n\\n\\ndef _embed_text(rec: EntityRec) -> str:\\n    return f\\"{rec.name}: {rec.merged_description()}\\".strip()[:512]\\n\\n\\n# ------------------------------------------------------------- (de)serialize\\ndef _entity_to_dict(rec: EntityRec) -> Dict:\\n    return {\\n        \\"key\\": rec.key,\\n        \\"name\\": rec.name,\\n        \\"type\\": rec.type,\\n        \\"descriptions\\": rec.descriptions,\\n        \\"chunk_ids\\": sorted(rec.chunk_ids),\\n        \\"summary\\": rec.summary,\\n    }\\n\\n\\ndef _entity_from_dict(d: Dict) -> EntityRec:\\n    return EntityRec(\\n        key=d[\\"key\\"], name=d[\\"name\\"], type=d[\\"type\\"],\\n        descriptions=list(d.get(\\"descriptions\\", [])),\\n        chunk_ids=set(d.get(\\"chunk_ids\\", [])),\\n        summary=d.get(\\"summary\\", \\"\\"),\\n    )\\n\\n\\ndef _rel_to_dict(r: RelRec) -> Dict:\\n    return {\\n        \\"src_key\\": r.src_key, \\"dst_key\\": r.dst_key,\\n        \\"descriptions\\": r.descriptions, \\"weight\\": r.weight,\\n        \\"chunk_ids\\": sorted(r.chunk_ids),\\n    }\\n\\n\\ndef _rel_from_dict(d: Dict) -> RelRec:\\n    return RelRec(\\n        src_key=d[\\"src_key\\"], dst_key=d[\\"dst_key\\"],\\n        descriptions=list(d.get(\\"descriptions\\", [])),\\n        weight=float(d.get(\\"weight\\", 0.0)),\\n        chunk_ids=set(d.get(\\"chunk_ids\\", [])),\\n    )\\n\\n\\n@dataclass\\nclass GraphRAGIndex:\\n    entities: Dict[str, EntityRec]\\n    relationships: List[RelRec]\\n    communities: Dict[str, int]          # entity key -> community id\\n    reports: Dict[int, Dict]             # community id -> report\\n    entity_order: List[str]              # entity keys aligned to entity_embeddings\\n    entity_embeddings: np.ndarray\\n    entity_store: EmbeddingStore\\n    graph: ig.Graph\\n    chunk_store: EmbeddingStore          # SHARED\\n    entity_chunks: Dict[str, List[str]]  # entity key -> chunk ids\\n    n_extract_failures: int = 0\\n\\n\\ndef _gdir(corpus_dir: str) -> str:\\n    return os.path.join(corpus_dir, \\"graphrag\\")\\n\\n\\ndef _paths(gdir: str) -> Dict[str, str]:\\n    return {name: os.path.join(gdir, f\\"{name}.json\\") for name in\\n            (\\"extraction\\", \\"entities\\", \\"relationships\\", \\"communities\\", \\"reports\\")} | {\\n        \\"graph\\": os.path.join(gdir, \\"graph.graphml\\"),\\n    }\\n\\n\\ndef _exists(gdir: str) -> bool:\\n    p = _paths(gdir)\\n    return all(os.path.isfile(p[k]) for k in (\\"entities\\", \\"relationships\\", \\"communities\\", \\"reports\\", \\"graph\\"))\\n\\n\\ndef _fetch_embeddings(store: EmbeddingStore, texts: List[str]) -> np.ndarray:\\n    if not texts:\\n        return np.zeros((0, 1), dtype=np.float32)\\n    ids = [compute_mdhash_id(t, prefix=f\\"{_GR_NAMESPACE}-\\") for t in texts]\\n    return store.get_embeddings(ids)\\n\\n\\ndef build_or_load(\\n    poc_cfg, bench_cfg, corpus_dir: str, chunk_store: EmbeddingStore,\\n    emb: EmbeddingModel, llm, tracker, force: bool = False,\\n) -> GraphRAGIndex:\\n    gdir = _gdir(corpus_dir)\\n    os.makedirs(gdir, exist_ok=True)\\n    p = _paths(gdir)\\n    entity_store = EmbeddingStore(emb, gdir, _GR_NAMESPACE)\\n\\n    if _exists(gdir) and not force:\\n        logger.info(\\"Loading existing GraphRAG index\\")\\n        return _load(gdir, entity_store, chunk_store)\\n\\n    # --- extraction (checkpointed) ---\\n    n_failures = 0\\n    if os.path.isfile(p[\\"extraction\\"]) and not force:\\n        logger.info(\\"Loading GraphRAG extraction checkpoint\\")\\n        with open(p[\\"extraction\\"], \\"r\\", encoding=\\"utf-8\\") as f:\\n            ck = json.load(f)\\n        entities = {k: _entity_from_dict(v) for k, v in ck[\\"entities\\"].items()}\\n        relationships = [_rel_from_dict(r) for r in ck[\\"relationships\\"]]\\n        n_failures = ck.get(\\"n_extract_failures\\", 0)\\n    else:\\n        chunk_texts = {\\n            cid: row[\\"content\\"] for cid, row in chunk_store.get_text_for_all_rows().items()\\n        }\\n        entities, relationships, n_failures = batch_extract(llm, chunk_texts, bench_cfg, tracker)\\n        summarize_entities(llm, entities, bench_cfg)\\n        with open(p[\\"extraction\\"], \\"w\\", encoding=\\"utf-8\\") as f:\\n            json.dump({\\n                \\"entities\\": {k: _entity_to_dict(v) for k, v in entities.items()},\\n                \\"relationships\\": [_rel_to_dict(r) for r in relationships],\\n                \\"n_extract_failures\\": n_failures,\\n            }, f)\\n\\n    # --- entity graph + communities ---\\n    graph = comm.build_graph(entities, relationships)\\n    community_map = comm.detect_communities(graph)\\n\\n    members: Dict[int, List[str]] = {}\\n    for key, cid in community_map.items():\\n        members.setdefault(cid, []).append(key)\\n\\n    rels_by_comm: Dict[int, List[RelRec]] = {}\\n    for r in relationships:\\n        c1, c2 = community_map.get(r.src_key), community_map.get(r.dst_key)\\n        if c1 is not None and c1 == c2:\\n            rels_by_comm.setdefault(c1, []).append(r)\\n\\n    reports: Dict[int, Dict] = {}\\n    for cid, keys in members.items():\\n        if len(keys) < bench_cfg.gr_min_community_size:\\n            continue\\n        reports[cid] = comm.community_report(\\n            llm, keys, entities, rels_by_comm.get(cid, []), bench_cfg\\n        )\\n    logger.info(\\"GraphRAG built %d community reports\\", len(reports))\\n\\n    # --- entity embeddings (\\"name: description\\") ---\\n    entity_order = list(entities.keys())\\n    embed_texts = [_embed_text(entities[k]) for k in entity_order]\\n    entity_store.insert_strings(embed_texts)\\n    entity_embeddings = _fetch_embeddings(entity_store, embed_texts)\\n\\n    entity_chunks = {k: sorted(entities[k].chunk_ids) for k in entity_order}\\n\\n    _persist(gdir, entities, relationships, community_map, reports,\\n             entity_order, embed_texts, entity_chunks, graph, n_failures)\\n\\n    return GraphRAGIndex(\\n        entities=entities, relationships=relationships, communities=community_map,\\n        reports=reports, entity_order=entity_order, entity_embeddings=entity_embeddings,\\n        entity_store=entity_store, graph=graph, chunk_store=chunk_store,\\n        entity_chunks=entity_chunks, n_extract_failures=n_failures,\\n    )\\n\\n\\ndef _persist(gdir, entities, relationships, community_map, reports, entity_order,\\n             embed_texts, entity_chunks, graph, n_failures) -> None:\\n    p = _paths(gdir)\\n    with open(p[\\"entities\\"], \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump({\\n            \\"entities\\": {k: _entity_to_dict(v) for k, v in entities.items()},\\n            \\"entity_order\\": entity_order,\\n            \\"embed_texts\\": embed_texts,\\n            \\"entity_chunks\\": entity_chunks,\\n            \\"n_extract_failures\\": n_failures,\\n        }, f)\\n    with open(p[\\"relationships\\"], \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump([_rel_to_dict(r) for r in relationships], f)\\n    with open(p[\\"communities\\"], \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(community_map, f)\\n    with open(p[\\"reports\\"], \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump({str(k): v for k, v in reports.items()}, f)\\n    graph.write_graphml(p[\\"graph\\"])\\n\\n\\ndef _load(gdir, entity_store, chunk_store) -> GraphRAGIndex:\\n    p = _paths(gdir)\\n    with open(p[\\"entities\\"], \\"r\\", encoding=\\"utf-8\\") as f:\\n        edata = json.load(f)\\n    entities = {k: _entity_from_dict(v) for k, v in edata[\\"entities\\"].items()}\\n    entity_order = edata[\\"entity_order\\"]\\n    embed_texts = edata[\\"embed_texts\\"]\\n    entity_chunks = edata[\\"entity_chunks\\"]\\n    n_failures = edata.get(\\"n_extract_failures\\", 0)\\n    with open(p[\\"relationships\\"], \\"r\\", encoding=\\"utf-8\\") as f:\\n        relationships = [_rel_from_dict(r) for r in json.load(f)]\\n    with open(p[\\"communities\\"], \\"r\\", encoding=\\"utf-8\\") as f:\\n        community_map = json.load(f)\\n    with open(p[\\"reports\\"], \\"r\\", encoding=\\"utf-8\\") as f:\\n        reports = {int(k): v for k, v in json.load(f).items()}\\n    graph = ig.Graph.Read_GraphML(p[\\"graph\\"])\\n    entity_embeddings = _fetch_embeddings(entity_store, embed_texts)\\n    return GraphRAGIndex(\\n        entities=entities, relationships=relationships, communities=community_map,\\n        reports=reports, entity_order=entity_order, entity_embeddings=entity_embeddings,\\n        entity_store=entity_store, graph=graph, chunk_store=chunk_store,\\n        entity_chunks=entity_chunks, n_extract_failures=n_failures,\\n    )\\n", "benchmark/graphrag/prompts.py": "\\"\\"\\"GraphRAG prompts: entity/relationship extraction, description summarization,\\nand community reports. All emit JSON (``json_mode=True``), kept short with one\\nworked example so a CPU-served 20B model stays parseable.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Dict, List\\n\\n# ------------------------------------------------------------------ extraction\\n_EXTRACT_SYSTEM = \\"\\"\\"You are a knowledge-graph extractor. From the text, extract\\nnamed entities and the relationships between them.\\n\\nReturn ONLY a JSON object with this exact shape:\\n{{\\"entities\\": [{{\\"name\\": \\"...\\", \\"type\\": \\"...\\", \\"description\\": \\"...\\"}}],\\n \\"relationships\\": [{{\\"source\\": \\"...\\", \\"target\\": \\"...\\", \\"description\\": \\"...\\", \\"strength\\": 1-10}}]}}\\n\\nRules:\\n- \\"type\\" must be one of: {entity_types}.\\n- \\"name\\" is the entity as it appears (proper noun / phrase).\\n- \\"description\\" is a short factual phrase grounded in the text.\\n- \\"source\\" and \\"target\\" of every relationship must be entity names you also list.\\n- \\"strength\\" is an integer 1-10 rating how strongly the text ties the two entities.\\nExtract nothing that is not supported by the text.\\"\\"\\"\\n\\n_EXTRACT_EXAMPLE = \\"\\"\\"Example.\\nText: \\"Marie Curie, a physicist, was born in Warsaw and won the Nobel Prize.\\"\\nJSON:\\n{\\"entities\\": [\\n  {\\"name\\": \\"Marie Curie\\", \\"type\\": \\"person\\", \\"description\\": \\"physicist born in Warsaw\\"},\\n  {\\"name\\": \\"Warsaw\\", \\"type\\": \\"location\\", \\"description\\": \\"birthplace of Marie Curie\\"},\\n  {\\"name\\": \\"Nobel Prize\\", \\"type\\": \\"work\\", \\"description\\": \\"award won by Marie Curie\\"}],\\n \\"relationships\\": [\\n  {\\"source\\": \\"Marie Curie\\", \\"target\\": \\"Warsaw\\", \\"description\\": \\"was born in\\", \\"strength\\": 8},\\n  {\\"source\\": \\"Marie Curie\\", \\"target\\": \\"Nobel Prize\\", \\"description\\": \\"won the\\", \\"strength\\": 9}]}\\"\\"\\"\\n\\n_EXTRACT_USER = \\"\\"\\"{example}\\n\\nNow extract from this text.\\nText:\\n```\\n{text}\\n```\\nJSON:\\"\\"\\"\\n\\n\\ndef extraction_messages(text: str, entity_types: List[str]) -> List[Dict[str, str]]:\\n    system = _EXTRACT_SYSTEM.format(entity_types=\\", \\".join(entity_types))\\n    return [\\n        {\\"role\\": \\"system\\", \\"content\\": system},\\n        {\\"role\\": \\"user\\", \\"content\\": _EXTRACT_USER.format(example=_EXTRACT_EXAMPLE, text=text)},\\n    ]\\n\\n\\n_GLEAN_USER = \\"\\"\\"Some entities and relationships were missed in the previous\\nextraction of the same text. Add ONLY new ones, using the exact same JSON shape\\n{{\\"entities\\": [...], \\"relationships\\": [...]}}. Do not repeat anything already\\nlisted below.\\n\\nText:\\n```\\n{text}\\n```\\n\\nAlready extracted:\\n{prev_json}\\n\\nAdditional JSON:\\"\\"\\"\\n\\n\\ndef gleaning_messages(text: str, prev_json: str, entity_types: List[str]) -> List[Dict[str, str]]:\\n    system = _EXTRACT_SYSTEM.format(entity_types=\\", \\".join(entity_types))\\n    return [\\n        {\\"role\\": \\"system\\", \\"content\\": system},\\n        {\\"role\\": \\"user\\", \\"content\\": _GLEAN_USER.format(text=text, prev_json=prev_json)},\\n    ]\\n\\n\\n# ------------------------------------------------------- description summaries\\n_SUMMARIZE_SYSTEM = \\"\\"\\"You merge several descriptions of the same entity into one\\nconcise, factual description. Return ONLY JSON: {\\"description\\": \\"...\\"}.\\"\\"\\"\\n\\n_SUMMARIZE_USER = \\"\\"\\"Entity: {name}\\nDescriptions:\\n{descriptions}\\n\\nMerged description as JSON:\\"\\"\\"\\n\\n\\ndef summarize_descriptions_messages(name: str, descriptions: List[str]) -> List[Dict[str, str]]:\\n    body = \\"\\\\n\\".join(f\\"- {d}\\" for d in descriptions)\\n    return [\\n        {\\"role\\": \\"system\\", \\"content\\": _SUMMARIZE_SYSTEM},\\n        {\\"role\\": \\"user\\", \\"content\\": _SUMMARIZE_USER.format(name=name, descriptions=body)},\\n    ]\\n\\n\\n# ----------------------------------------------------------- community reports\\n_REPORT_SYSTEM = \\"\\"\\"You write a concise analytical report about a community of\\nrelated entities. Return ONLY JSON:\\n{\\"title\\": \\"...\\", \\"summary\\": \\"...\\", \\"rating\\": 0-10,\\n \\"findings\\": [{\\"summary\\": \\"...\\", \\"explanation\\": \\"...\\"}]}\\n\\"rating\\" is the community\'s overall importance. Keep the summary under 120 words\\nand base everything strictly on the provided entities and relationships.\\"\\"\\"\\n\\n_REPORT_USER = \\"\\"\\"Entities:\\n{entities_block}\\n\\nRelationships:\\n{rels_block}\\n\\nWrite the community report as JSON:\\"\\"\\"\\n\\n\\ndef community_report_messages(entities_block: str, rels_block: str) -> List[Dict[str, str]]:\\n    return [\\n        {\\"role\\": \\"system\\", \\"content\\": _REPORT_SYSTEM},\\n        {\\"role\\": \\"user\\", \\"content\\": _REPORT_USER.format(\\n            entities_block=entities_block, rels_block=rels_block)},\\n    ]\\n", "benchmark/graphrag/search.py": "\\"\\"\\"GraphRAG LOCAL search: query -> entity match (by embedding) -> chunk scoring.\\n\\nRetrieval-comparable with the other systems (Recall@k is computed from\\n``retrieve()`` which returns documents only). ``build_qa_context`` assembles the\\nfaithful local-search context - community reports + relationship lines + chunks -\\nwithin the same ``qa_top_k`` budget used by BaseRAG/PropRAG.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nfrom typing import Dict, List, Tuple\\n\\nimport numpy as np\\n\\nfrom proprag_poc.core.retriever import RetrievedPassage, _min_max\\n\\nfrom .index import GraphRAGIndex\\n\\nlogger = logging.getLogger(__name__)\\n\\n\\nclass GraphRAGLocalRetriever:\\n    system_name = \\"GraphRAG\\"\\n\\n    def __init__(self, index: GraphRAGIndex, embedding_model, poc_cfg, bench_cfg):\\n        self.index = index\\n        self.embedding_model = embedding_model\\n        self.poc_cfg = poc_cfg\\n        self.cfg = bench_cfg\\n\\n        self.passage_keys = index.chunk_store.get_all_ids()\\n        self.passage_row = {k: i for i, k in enumerate(self.passage_keys)}\\n        self.passage_embeddings = index.chunk_store.get_embeddings(self.passage_keys)\\n\\n        self.entity_row = {k: i for i, k in enumerate(index.entity_order)}\\n        self.rel_lookup: Dict[frozenset, object] = {\\n            frozenset((r.src_key, r.dst_key)): r for r in index.relationships\\n        }\\n\\n    # ------------------------------------------------------------- helpers\\n    def _embed_query(self, query: str) -> np.ndarray:\\n        return self.embedding_model.batch_encode(\\n            query, instruction=self.poc_cfg.embedding_query_instruction,\\n            norm=True, input_type=\\"query\\",\\n        ).squeeze()\\n\\n    def _match_entities(self, q: np.ndarray) -> List[Tuple[str, float]]:\\n        if self.index.entity_embeddings.shape[0] == 0 or self.index.entity_embeddings.ndim != 2:\\n            return []\\n        sims = self.index.entity_embeddings @ q.T\\n        order = np.argsort(sims)[::-1][: self.cfg.gr_local_top_entities]\\n        out = []\\n        for i in order:\\n            score = float(sims[i])\\n            if score < self.cfg.gr_entity_sim_floor:\\n                break\\n            out.append((self.index.entity_order[i], score))\\n        return out\\n\\n    def _dense_scores(self, q: np.ndarray) -> np.ndarray:\\n        return np.atleast_1d((self.passage_embeddings @ q.T).squeeze())\\n\\n    # ------------------------------------------------------------- retrieve\\n    def retrieve(self, query: str, top_k: int = None) -> List[RetrievedPassage]:\\n        top_k = top_k or self.poc_cfg.retrieval_top_k\\n        q = self._embed_query(query)\\n        dense = _min_max(self._dense_scores(q))\\n        matched = self._match_entities(q)\\n\\n        if not matched:\\n            order = np.argsort(dense)[::-1][:top_k]\\n            return [self._passage(i, float(dense[i])) for i in order]\\n\\n        graph_score = np.zeros(len(self.passage_keys))\\n        for ekey, sim in matched:\\n            for cid in self.index.entity_chunks.get(ekey, []):\\n                row = self.passage_row.get(cid)\\n                if row is not None:\\n                    graph_score[row] += sim\\n\\n        # Bonus for chunks cited by a relationship between two matched entities.\\n        matched_keys = [k for k, _ in matched]\\n        for a in range(len(matched_keys)):\\n            for b in range(a + 1, len(matched_keys)):\\n                rel = self.rel_lookup.get(frozenset((matched_keys[a], matched_keys[b])))\\n                if rel is None:\\n                    continue\\n                for cid in rel.chunk_ids:\\n                    row = self.passage_row.get(cid)\\n                    if row is not None:\\n                        graph_score[row] += 0.5\\n\\n        if graph_score.sum() == 0:\\n            order = np.argsort(dense)[::-1][:top_k]\\n            return [self._passage(i, float(dense[i])) for i in order]\\n\\n        final = _min_max(graph_score) + self.cfg.gr_dense_blend * dense\\n        order = np.argsort(final)[::-1][:top_k]\\n        return [self._passage(i, float(final[i])) for i in order]\\n\\n    def _passage(self, row: int, score: float) -> RetrievedPassage:\\n        key = self.passage_keys[row]\\n        return RetrievedPassage(\\n            chunk_id=key,\\n            text=self.index.chunk_store.get_row(key)[\\"content\\"],\\n            score=score,\\n        )\\n\\n    # ---------------------------------------------------------- QA context\\n    def build_qa_context(self, query: str, passages: List[RetrievedPassage]) -> List[str]:\\n        \\"\\"\\"Local-search context within the qa_top_k budget: community reports +\\n        relationship lines + top chunks.\\"\\"\\"\\n        budget = self.cfg.qa_top_k\\n        q = self._embed_query(query)\\n        matched = self._match_entities(q)\\n        matched_keys = [k for k, _ in matched]\\n\\n        context: List[str] = []\\n\\n        # 1-2 community reports covering the matched entities (highest rating first).\\n        seen_comm = set()\\n        comm_reports = []\\n        for ekey in matched_keys:\\n            cid = self.index.communities.get(ekey)\\n            if cid is None or cid in seen_comm:\\n                continue\\n            report = self.index.reports.get(int(cid))\\n            if report:\\n                seen_comm.add(cid)\\n                comm_reports.append(report)\\n        comm_reports.sort(key=lambda r: r.get(\\"rating\\", 0), reverse=True)\\n        for report in comm_reports[:2]:\\n            context.append(f\\"[Community report] {report.get(\'title\', \'\')}: {report.get(\'summary\', \'\')}\\")\\n\\n        # Relationship one-liners between matched entities.\\n        rel_lines = []\\n        for a in range(len(matched_keys)):\\n            for b in range(a + 1, len(matched_keys)):\\n                rel = self.rel_lookup.get(frozenset((matched_keys[a], matched_keys[b])))\\n                if rel is None:\\n                    continue\\n                na = self.index.entities[rel.src_key].name\\n                nb = self.index.entities[rel.dst_key].name\\n                desc = \\" ; \\".join(dict.fromkeys(rel.descriptions)) or \\"related\\"\\n                rel_lines.append(f\\"{na} -> {nb}: {desc}\\")\\n        if rel_lines:\\n            context.append(\\"[Relationships]\\\\n\\" + \\"\\\\n\\".join(rel_lines[:10]))\\n\\n        # Fill the remaining budget with top retrieved chunks.\\n        for p in passages:\\n            if len(context) >= budget:\\n                break\\n            context.append(p.text)\\n        return context[:budget] if context else [p.text for p in passages[:budget]]\\n"}'
files_dict = json.loads(serialized_files)

for path, content in files_dict.items():
    full_path = "/content/" + path
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Wrote {full_path} ({len(content)} chars)")
print("All benchmark files written successfully to Colab disk!")

## Section 5 — Download Dataset
> Checks if the dataset files exist locally (e.g. from git clone) and copies them, otherwise downloads from HuggingFace Datasets.

In [ ]:
import os, json, shutil

DATA_DIR    = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

REPO_DS_DIR  = os.path.join(PROPRAG_MAIN, "reproduce", "dataset")
DATASET_PATH = os.path.join(DATA_DIR, "2wikimultihopqa.json")
CORPUS_PATH  = os.path.join(DATA_DIR, "2wikimultihopqa_corpus.json")

def _try_copy(basename, dst):
    src = os.path.join(REPO_DS_DIR, basename)
    if os.path.isfile(dst):
        print(f"  Already present: {dst}")
        return True
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f"  Copied from repository: {src}")
        return True
    return False

print("Locating 2WikiMultiHopQA dataset files...")
have_q = _try_copy("2wikimultihopqa.json",        DATASET_PATH)
have_c = _try_copy("2wikimultihopqa_corpus.json",  CORPUS_PATH)

if not (have_q and have_c):
    print("Downloading from HuggingFace Datasets (voidful/2WikiMultiHopQA)...")
    from datasets import load_dataset
    ds = load_dataset("voidful/2WikiMultiHopQA", trust_remote_code=True)

    if not have_q:
        split = ds.get("validation") or ds.get("train")
        records = []
        for ex in split:
            records.append({
                "_id":             ex.get("id", ex.get("_id", str(len(records)))),
                "type":            ex.get("type", "compositional"),
                "question":        ex["question"],
                "answer":          ex["answer"],
                "supporting_facts": [[sf["title"], sf.get("sent_id", 0)]
                                     if isinstance(sf, dict) else sf
                                     for sf in ex.get("supporting_facts", [])],
                "context": [[c["title"], c.get("sentences", [])]
                            if isinstance(c, dict) else c
                            for c in ex.get("context", [])],
            })
        with open(DATASET_PATH, "w", encoding="utf-8") as f:
            json.dump(records, f)
        print(f"  Dataset saved: {len(records)} questions")

    if not have_c:
        seen, corpus = set(), []
        for sname in ds:
            for ex in ds[sname]:
                for ctx in ex.get("context", []):
                    title = ctx["title"] if isinstance(ctx, dict) else ctx[0]
                    sents = (ctx.get("sentences", []) if isinstance(ctx, dict)
                             else (ctx[1] if len(ctx) > 1 else []))
                    if title not in seen:
                        seen.add(title)
                        corpus.append({"title": title, "text": " ".join(sents) if sents else ""})
        with open(CORPUS_PATH, "w", encoding="utf-8") as f:
            json.dump(corpus, f)
        print(f"  Corpus saved: {len(corpus)} documents")

os.environ["BENCH_DATASET_PATH"] = DATASET_PATH
os.environ["BENCH_CORPUS_PATH"]  = CORPUS_PATH
print(f"\nDataset path : {DATASET_PATH}")
print(f"Corpus path  : {CORPUS_PATH}")

## Section 6 — Load `openai/gpt-oss-20b` with 4-bit NF4 Quantization
> Downloads weights (~20 GB on first run — takes 10-20 min).
> **Ensure you have accepted the model license** at https://huggingface.co/openai/gpt-oss-20b first.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

_hf_kw = {"token": HF_TOKEN} if HF_TOKEN else {}

print(f"Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **_hf_kw)
print("Tokenizer loaded successfully.")

_compute_dtype = torch.float16 if BNB_COMPUTE_DTYPE == "float16" else torch.bfloat16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=BNB_QUANT_TYPE,
    bnb_4bit_compute_dtype=_compute_dtype,
    bnb_4bit_use_double_quant=BNB_DOUBLE_QUANT,
)

print(f"\nLoading {MODEL_ID} in 4-bit NF4 (bitsandbytes)...")
print("  Downloading ~20 GB from HuggingFace. Please wait.")

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="eager",  # Workaround: gpt-oss uses custom attention layers
        trust_remote_code=True,
        **_hf_kw,
    )
    model.eval()
    print("\n✅ Model loaded successfully!")
    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  GPU VRAM: {used:.1f} GB / {total:.1f} GB")
except Exception as e:
    print(f"\n❌ Model loading failed: {e}")
    print("\nTroubleshooting:")
    print("  1. Go to https://huggingface.co/openai/gpt-oss-20b and make sure you accepted the license terms.")
    print("  2. Paste your HuggingFace Token in Cell 3 (Configuration).")
    raise

## Section 7 — Start OpenAI-Compatible API Server
> Runs a FastAPI server in a background thread on `http://127.0.0.1:5001/v1`. Keep special tokens in model output so that Harmony reasoning stripping works.

In [ ]:
import threading, time, uuid, asyncio
import torch
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional, Any

app = FastAPI(title="gpt-oss-20b API server")

class _Msg(BaseModel):
    role: str
    content: str

class _ChatReq(BaseModel):
    model:           str
    messages:        List[_Msg]
    max_tokens:      Optional[int]   = 512
    temperature:     Optional[float] = 0.0
    stream:          Optional[bool]  = False
    response_format: Optional[Any]   = None
    seed:            Optional[int]   = None

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [
        {"id": MODEL_ID, "object": "model", "created": 0, "owned_by": "openai"}
    ]}

@app.post("/v1/chat/completions")
async def chat_completions(req: _ChatReq):
    messages    = [{"role": m.role, "content": m.content} for m in req.messages]
    max_new     = min(req.max_tokens or 512, MAX_NEW_TOKENS_DEFAULT)
    do_sample   = (req.temperature or 0.0) > 0.01
    temperature = req.temperature if do_sample else 1.0

    with torch.inference_mode():
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                reasoning_effort="medium",
            )
        except Exception:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )

        inputs = tokenizer(
            text, return_tensors="pt",
            truncation=True, max_length=6144,
        ).to(model.device)

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=do_sample,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id or tokenizer.pad_token_id or 0,
            eos_token_id=tokenizer.eos_token_id,
        )

    n_prompt = inputs["input_ids"].shape[1]
    new_ids  = output_ids[0][n_prompt:]
    raw_text = tokenizer.decode(new_ids, skip_special_tokens=False,
                                clean_up_tokenization_spaces=True)

    return {
        "id":      f"chatcmpl-{uuid.uuid4().hex[:8]}",
        "object":  "chat.completion",
        "created": int(time.time()),
        "model":   MODEL_ID,
        "choices": [{"index": 0,
                     "message": {"role": "assistant", "content": raw_text},
                     "finish_reason": "stop"}],
        "usage":   {"prompt_tokens":     n_prompt,
                    "completion_tokens": int(len(new_ids)),
                    "total_tokens":      n_prompt + int(len(new_ids))},
    }

# Start uvicorn
_cfg    = uvicorn.Config(app, host=LLAMA_HOST, port=LLAMA_PORT,
                         log_level="warning", loop="asyncio")
_server = uvicorn.Server(_cfg)
_thread = threading.Thread(target=_server.run, daemon=True)
_thread.start()

# Wait up to 15 seconds
import urllib.request, urllib.error
_url = f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1/models"
for _i in range(30):
    time.sleep(0.5)
    try:
        with urllib.request.urlopen(_url, timeout=3): break
    except Exception: pass
else:
    raise RuntimeError("FastAPI server did not start within 15 seconds.")

print(f"✅ OpenAI-compatible server running at http://{LLAMA_HOST}:{LLAMA_PORT}/v1")

## Section 8 — Verify End-to-End (Smoke Test)

In [ ]:
from openai import OpenAI
from benchmark.llm_wrap import strip_gpt_oss_reasoning

client = OpenAI(
    base_url=f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1",
    api_key="not-needed",
)

resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one word, what is 2 + 2?"}],
    max_tokens=64,
    temperature=0.0,
)
raw      = resp.choices[0].message.content
stripped = strip_gpt_oss_reasoning(raw)
print(f"Raw output  : {raw!r}")
print(f"After strip : {stripped!r}")
print("\n✅ Smoke test passed!")

## Section 9 — Initialize Benchmark

In [ ]:
import sys, os, logging, json

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# Reload to make sure we load newly written files
for mod in list(sys.modules):
    if mod.startswith("benchmark"):
        del sys.modules[mod]

from proprag_poc.logging_setup import setup_logging
setup_logging()

_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(logging.Formatter(
    "%(asctime)s | %(levelname)-5s | %(name)s | %(message)s", "%%H:%%M:%%S"
))
bench_log = logging.getLogger("benchmark")
bench_log.setLevel(logging.INFO)
bench_log.handlers.clear()
bench_log.addHandler(_handler)
bench_log.propagate = False

from benchmark.bench_config         import BenchmarkConfig
from benchmark.llm_wrap             import BenchLLMClient, check_backend
from benchmark.dataset              import (build_corpus, corpus_id, load_questions,
                                            pilot_subset, stratified_subset,
                                            subset_hash, write_manifest, type_counts)
from benchmark.evaluation           import em_score, f1_score, recall_at_k
from benchmark.results              import ResultsStore
from benchmark.usage               import IndexPhase, delta
from benchmark                      import proprag_adapter as adapter
from benchmark                      import report as report_mod
from benchmark.graphrag             import index as gr_index_mod
from benchmark.graphrag.search      import GraphRAGLocalRetriever
from benchmark.qa                   import answer_question
from proprag_poc.core.metrics       import get_usage_tracker
from proprag_poc.embedding.encoder  import EmbeddingModel

cfg = BenchmarkConfig(
    project_dir  = "/content",
    dataset_path = DATASET_PATH,
    corpus_path  = CORPUS_PATH,
    n_questions  = N_QUESTIONS,
    seed         = SEED,
    pilot        = PILOT,
)
poc_cfg = cfg.make_poc_config()
systems = [s for s in SYSTEMS if s in ("BaseRAG", "GraphRAG", "PropRAG")]

check_backend(poc_cfg)
print("✅ Setup verified.")

In [ ]:
import json, os

all_qs    = load_questions(cfg.dataset_path)
subset    = stratified_subset(all_qs, cfg.n_questions, cfg.seed)
questions = pilot_subset(subset, cfg.pilot, cfg.seed) if cfg.pilot else subset

print(f"Loaded: {len(all_qs)} total | subset={len(subset)} | active={len(questions)}")
print(f"Type counts: {type_counts(questions)}")

titles       = build_corpus(questions, cfg.corpus_path)
docs         = [(t, titles[t]) for t in sorted(titles)]
sub_hash     = subset_hash(questions)
tag          = f"n{cfg.n_questions}" + (f"_pilot{cfg.pilot}" if cfg.pilot else "")
corpus_ident = corpus_id(questions, tag)
run_id       = f"{cfg.n_questions}q_seed{cfg.seed}" + (f"_pilot{cfg.pilot}" if cfg.pilot else "")
run_dir      = os.path.join(cfg.data_dir, "benchmark", run_id)
os.makedirs(run_dir, exist_ok=True)

write_manifest(run_dir, questions, titles, cfg.seed, cfg)

meta_path = os.path.join(run_dir, "run_meta.json")
meta = {
    "subset_hash": sub_hash, "seed": cfg.seed, "n_questions": cfg.n_questions,
    "pilot": cfg.pilot, "llm_backend": poc_cfg.llm_backend,
    "llm_model": poc_cfg.llm_model, "embedding_model": poc_cfg.embedding_model,
    "qa_top_k": cfg.qa_top_k, "retrieval_top_k": cfg.retrieval_top_k,
    "gr_max_gleanings": cfg.gr_max_gleanings,
}
if os.path.isfile(meta_path):
    with open(meta_path) as f: prev = json.load(f)
    if prev.get("subset_hash") != sub_hash:
        raise RuntimeError(
            f"Refusing to resume: existing run used subset {prev['subset_hash']!r}, "
            f"current is {sub_hash!r}. Change seed/n_questions or delete {run_dir}."
        )
with open(meta_path, "w") as f: json.dump(meta, f, indent=2)

print(f"\nCorpus ready: {len(docs)} documents | id={corpus_ident}")
print(f"Run dir      : {run_dir}")

## Section 10 — Indexing Phase
> Builds BaseRAG, PropRAG, and GraphRAG indexes. Cached and resume-safe.

In [ ]:
import time, json, os

emb     = EmbeddingModel(poc_cfg)
llm     = BenchLLMClient(poc_cfg)
tracker = get_usage_tracker()
cdir    = adapter.corpus_dir(poc_cfg, corpus_ident)
usage_path  = os.path.join(run_dir, "index_usage.json")
index_usage = {}

bench_log.info("Indexing %d systems over %d docs (corpus %s)",
               len(systems), len(docs), corpus_ident)

# ── BaseRAG ──────────────────────────────────────────────────────────────────
bench_log.info("=== BaseRAG index ===")
with IndexPhase(tracker, "BaseRAG") as phase:
    chunk_store, chunk_id_to_title = adapter.build_base_index(
        poc_cfg, corpus_ident, docs, emb)
index_usage["BaseRAG"] = phase.usage
bench_log.info("BaseRAG done: %.1f s", phase.usage.get("wall_time_s", 0))

# ── PropRAG ──────────────────────────────────────────────────────────────────
bench_log.info("=== PropRAG index ===")
with IndexPhase(tracker, "PropRAG") as phase:
    corpus = adapter.build_or_load_proprag(
        poc_cfg, corpus_ident, chunk_store, chunk_id_to_title,
        emb, llm, force=FORCE_REINDEX)
index_usage["PropRAG"] = phase.usage
bench_log.info("PropRAG done: %.1f s", phase.usage.get("wall_time_s", 0))

# ── GraphRAG ─────────────────────────────────────────────────────────────────
bench_log.info("=== GraphRAG index ===")
with IndexPhase(tracker, "GraphRAG", extra_scopes=["index::GraphRAG"]) as phase:
    gr_index = gr_index_mod.build_or_load(
        poc_cfg, cfg, cdir, chunk_store, emb, llm, tracker,
        force=FORCE_REINDEX)
gr_usage = dict(phase.usage)
gr_usage["parse_failures"] = gr_index.n_extract_failures
index_usage["GraphRAG"] = gr_usage
bench_log.info("GraphRAG done: %.1f s | %d entities | %d communities",
               gr_usage.get("wall_time_s", 0),
               len(gr_index.entities),
               len(set(gr_index.communities.values())))

with open(usage_path, "w") as f: json.dump(index_usage, f, indent=2)

print("\nAll indexes loaded:")
for s, u in index_usage.items():
    print(f"  {s:10s}: wall={u.get('wall_time_s',0):6.1f}s  "
          f"prompt_tok={u.get('chat_prompt_tokens',0):6.0f}  "
          f"compl_tok={u.get('chat_completion_tokens',0):6.0f}")

## Section 11 — Question Loop
> Runs retrieval and generation for each system. Crash-safe and resume-safe.

In [ ]:
import time, json

retrievers = {}
if "BaseRAG"  in systems: retrievers["BaseRAG"]  = adapter.make_baserag_retriever(corpus, emb, poc_cfg)
if "PropRAG"  in systems: retrievers["PropRAG"]  = adapter.make_proprag_retriever(corpus, emb, poc_cfg)
if "GraphRAG" in systems: retrievers["GraphRAG"] = GraphRAGLocalRetriever(gr_index, emb, poc_cfg, cfg)

store = ResultsStore(run_dir)
done  = store.done_keys()
per_sys_secs = {s: [] for s in systems}

t_start = time.monotonic()

for qi, q in enumerate(questions, 1):
    bench_log.info("Q %d/%d [%s] %s", qi, len(questions), q.qtype, q.question[:80])
    for system in systems:
        if (q.qid, system) in done:
            continue
        retriever = retrievers[system]
        scope     = f"q::{system}"
        before    = tracker.snapshot(scope).as_dict()
        try:
            t0 = time.monotonic()
            with tracker.scope(scope):
                passages = retriever.retrieve(q.question, top_k=poc_cfg.retrieval_top_k)
                ret_lat  = time.monotonic() - t0

                ret_titles = [chunk_id_to_title.get(p.chunk_id, "") for p in passages]
                recall     = recall_at_k(q.gold_titles, ret_titles, cfg.recall_ks)

                if system == "GraphRAG":
                    context = retriever.build_qa_context(q.question, passages)
                else:
                    context = [p.text for p in passages[:cfg.qa_top_k]]

                t1 = time.monotonic()
                answer, raw = answer_question(llm, q.question, context, cfg)
                qa_lat = time.monotonic() - t1

            usage = delta(before, tracker.snapshot(scope).as_dict())
            row = {
                "qid": q.qid, "qtype": q.qtype, "system": system,
                "question": q.question, "gold_answer": q.answer,
                "gold_titles": q.gold_titles, "answer": answer, "raw_answer": raw,
                "retrieved": [{"chunk_id": p.chunk_id,
                               "title": chunk_id_to_title.get(p.chunk_id, ""),
                               "score": p.score} for p in passages],
                "recall":              recall,
                "em":                  em_score(q.gold_answers, answer),
                "f1":                  f1_score(q.gold_answers, answer),
                "retrieval_latency_s": round(ret_lat, 3),
                "qa_latency_s":        round(qa_lat,  3),
                "usage":               usage,
                "ts":                  time.time(),
            }
            store.append(row)
            per_sys_secs[system].append(ret_lat + qa_lat)
            bench_log.info("  -> %s | em=%.0f f1=%.2f r@5=%.2f | %.1fs",
                           system, row["em"], row["f1"],
                           recall.get("recall@5", 0), ret_lat + qa_lat)
        except Exception as _e:
            bench_log.exception("Q %s system %s failed", q.qid, system)
            store.append({"qid": q.qid, "qtype": q.qtype, "system": system,
                          "error": str(_e), "ts": time.time()})

wall = time.monotonic() - t_start
print(f"\nQuestion loop complete. Wall: {wall/60:.1f} min")
if cfg.pilot:
    newly = [s for secs in per_sys_secs.values() for s in secs]
    if newly:
        mean_q = sum(newly) / len(newly)
        proj   = mean_q * len(systems) * cfg.n_questions
        print(f"Pilot: mean {mean_q:.1f}s/call -> projected query loop wall ~{proj/60:.0f} min")

## Section 12 — Build Report & Display Results

In [ ]:
metrics     = report_mod.build(run_dir, make_charts=MAKE_CHARTS)
report_path = os.path.join(run_dir, "report.md")
print(f"Report saved to: {report_path}")

In [ ]:
from IPython.display import Markdown, display

with open(report_path, "r", encoding="utf-8") as f:
    report_md = f.read()

display(Markdown(report_md))

In [ ]:
import os, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

charts_dir = os.path.join(run_dir, "charts")
if os.path.isdir(charts_dir):
    chart_files = sorted(f for f in os.listdir(charts_dir) if f.endswith(".png"))
    if chart_files:
        fig, axes = plt.subplots(1, len(chart_files), figsize=(7 * len(chart_files), 5))
        if len(chart_files) == 1:
            axes = [axes]
        for ax, fname in zip(axes, chart_files):
            img = mpimg.imread(os.path.join(charts_dir, fname))
            ax.imshow(img); ax.axis("off")
            ax.set_title(fname.replace(".png","").replace("_"," ").title())
        plt.tight_layout()
        plt.show()
    else:
        print("No charts found.")
else:
    print("Charts directory not found.")

In [ ]:
import json, os
import pandas as pd
from IPython.display import display

metrics_path = os.path.join(run_dir, "metrics.json")
if os.path.isfile(metrics_path):
    with open(metrics_path) as f: m = json.load(f)
    rows = []
    for sys_name in ["BaseRAG", "GraphRAG", "PropRAG"]:
        if sys_name not in m: continue
        s = m[sys_name]
        rows.append({
            "System":         sys_name,
            "EM":             f"{s['em']*100:.1f}%",
            "F1":             f"{s['f1']*100:.1f}%",
            "Recall@5":       f"{s.get('recall@5',0)*100:.1f}%",
            "Recall@10":      f"{s.get('recall@10',0)*100:.1f}%",
            "Chat calls/Q":   f"{s['mean_chat_calls_per_q']:.2f}",
            "QA latency (s)": f"{s['mean_qa_latency_s']:.2f}",
        })
    df = pd.DataFrame(rows).set_index("System")
    display(df.style.set_caption("Summary Metrics Table"))